In [1]:
# Parameters
BATCH_MODE = "true"

# Complexity-03j — Deux messages suffisent : `QIP(2)` et le sum-check compressé

**Série Complexity — accrétion de 03** · Public : **Recherche** · Hommage : **Yupan Liu** (l'annonce du 23 novembre 2026) et **Rahul Jain, Zhengfeng Ji, Sarvagya Upadhyay & John Watrous** (la base établie, 2011)

**Navigation** : [Index de la série](README.md) | [03h — L'exposant du produit matriciel est 9/4](Complexity-03h-ExposantProduitMatriciel-NineQuarts-Python.ipynb) | [03k — L'arbre couvrant de poids minimal en temps linéaire déterministe](Complexity-03k-KhannaPuttermanSong-ArbreCouvrantLineaireDeterministe-Python.ipynb) | [03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-Python.ipynb) | [03c — Zoo et oracles](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) | [06 — Simuler un circuit quantique classiquement](Complexity-06-Simuler-Circuit-Quantique-Classiquement-Python.ipynb) | [06b — Aaronson & Gottesman, déquantification](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb)

> **Bandeau d'honnêteté.** Deux énoncés, deux statuts, jamais confondus. `QIP(3) = QIP = PSPACE` est **établi** depuis 2011 (Jain, Ji, Upadhyay & Watrous, JACM 58(6)) et **cité** ici, jamais rejoué. `QIP(2) = PSPACE` est **annoncé** le 23 novembre 2026 par un article d'un seul auteur, **non évalué par les pairs** : ce carnet n'en reproduit aucune preuve, ne le présente jamais comme établi, et ne prétend pas que la difficulté PSPACE de `CIUniform` est démontrée indépendamment. Ce que ce carnet **rejoue** est plus modeste et vérifiable ligne à ligne : le protocole du sum-check de Shamir sur un petit domaine, l'échantillonneur exact qui porte la transformation vers `CIUniform`, le vérificateur à deux messages réduit à ses amplitudes, et une matrice jouet qui montre ce que « stoquastique » interdit — donc ce qu'il coûte.

## 0. Pourquoi ce carnet

[03](Complexity-03-HierarchieTemporelle-Python.ipynb) a posé la hiérarchie en temps ; [03c](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) a navigué le Zoo jusqu'aux oracles ; [06](Complexity-06-Simuler-Circuit-Quantique-Classiquement-Python.ipynb) a simulé un circuit quantique porte par porte ; [06b](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb) a montré comment une famille de circuits se déquantifie. Ce carnet prend la question voisine, celle des **preuves interactives** : combien de messages faut-il au vérificateur pour convaincre, et combien la physique autorise-t-elle à en économiser ?

Côté classique, la réponse est un théorème : `IP = PSPACE` (Lund–Fortnow–Karloff–Nisan 1992 ; Shamir 1992), et le **sum-check** de Shamir en est la pièce maîtresse. Côté quantique, `QIP = QIP(3)` (Kitaev–Watrous, STOC 2000) puis `QIP = PSPACE` (Jain, Ji, Upadhyay & Watrous, JACM 2011) : **trois** messages suffisent, quel que soit le protocole de départ. Restait la question de savoir si **deux** suffisent. Le 23 novembre 2026, Yupan Liu annonce que oui : `qq-StoqAM = qq-QAM = QIP(2) = QIP = PSPACE`, avec un vérificateur qui n'envoie que des **moitiés de paires EPR** (« pièces publiques quantiques ») et ne fait ensuite qu'une **vérification stoquastique**.

Ce que le lecteur emporte : (1) le sum-check **rejoué** sur un domaine fini, avec le contrôle que la somme annoncée est correcte et la probabilité de triche mesurée par programme dynamique exact ; (2) la **différence structurelle** entre un protocole à deux et à trois messages, montrée par exécution sur un jouet ; (3) l'**échantillonneur** exact qui porte la transformation vers `CIUniform`, avec l'équation (3.5) vérifiée et la distance minimale en variation totale calculée par programme linéaire ; (4) le vérificateur à deux messages **simulé qubit par qubit**, confronté à deux formules indépendantes à 1e-16 près ; (5) « stoquastique » défini et **mesuré** : ce que le mot interdit, et ce que la contrainte coûte sur un jouet.

In [2]:
import functools
import itertools
import time
from collections.abc import Callable

import numpy as NP
from scipy.optimize import linprog
from scipy.sparse import csr_matrix, hstack, identity, vstack

# Toutes les experiences de ce carnet sont deterministes. Chaque bloc aleatoire tire sur un generateur
# seede une fois, dans un ordre fige -- memes tirages, meme execution d'une fois sur l'autre.
# `generateur` sert aux sections 3 et 4 ; la section 2 a le sien, cree sur place.
SEED = 20261123  # date de depot du preprint distille ici
generateur = NP.random.default_rng(SEED)
CHRONO = time.perf_counter()

print(f"Imports OK : numpy {NP.__version__}, scipy (linprog, sparse)")
print(f"SEED = {SEED} -- toutes les instances aleatoires du carnet sont rejouables a l'identique.")

Imports OK : numpy 2.5.2, scipy (linprog, sparse)
SEED = 20261123 -- toutes les instances aleatoires du carnet sont rejouables a l'identique.


## 1. Le sum-check de Shamir, tour par tour

Le théorème `IP = PSPACE` repose sur un protocole à tours. Une formule quantifiée $Q_1 x_1 \cdots Q_m x_m\, \psi(x)$ devient un polynôme $P_T$ — l'extension multilinéaire de $\psi$ — que les quantificateurs transforment de proche en proche, $P_t = O_t P_{t+1}$, jusqu'à une constante égale à la valeur de vérité. À chaque tour, le prouveur annonce une restriction univariée $g(X)$, de degré individuel au plus deux. Le vérificateur, dans l'article de Liu (§1), « uses g(0) and g(1) to check the current claim $P_t(\alpha) = \nu$ according to the round-t operation », puis « evaluates g at a field element $r \in \mathbb{F}_q$ chosen uniformly at random if the check passes ». La solidité tient en une phrase, citée : « If the current claim is false and the check passes, then $g - g^\star$ is a nonzero polynomial of degree at most two and thus $g(r) = g^\star(r)$ for at most two values of $r \in \mathbb{F}_q$. Consequently, the next-round claim is correct with probability at most 2/q. » Une borne d'union sur les $T$ tours donne au plus $2T/q$.

Ce que ce carnet **mesure**, au lieu de le citer : que la somme annoncée est bien celle du tour (le contrôle, rejoué) ; la probabilité d'acceptation **exacte** du meilleur prouveur trompeur, calculée par programme dynamique sur tous ses messages possibles ; et l'endroit précis où un prouveur honnête échoue sur une formule fausse. Le modèle est un jouet : quatre formules à une ou deux variables, sur $\mathbb{F}_q$ pour $q$ premier petit. L'article, lui, prend $q = 8(T+1)$, une puissance de deux (Lemme 3.6) : ce choix du corps est le nôtre, il simplifie l'arithmétique, et rien ici ne transpose les constantes de l'article.

In [3]:
# --- 1a. Une instance : formule quantifiee sur F_q, operations, polynomes P_t ---
class Instance:
    """Formule Q_1 x_1 ... Q_m x_m psi(x) lue sur F_q, avec sa liste d'operations."""

    def __init__(self, q: int, psi, quantificateurs: tuple[str, ...]) -> None:
        self.q = q                                      # taille du corps F_q (premier ici)
        self.psi = psi                                  # psi(affectation booleenne) -> 0 ou 1
        self.quantificateurs = tuple(quantificateurs)   # "E" (il existe) ou "A" (pour tout)
        self.m = len(quantificateurs)
        # Lemme 3.5 : pour j = m, ..., 1 on applique R_1, ..., R_{j-1} puis Q_j
        operations: list[tuple[str, int]] = []
        for j in range(self.m, 0, -1):
            operations.extend(("R", k) for k in range(1, j))
            operations.append((self.quantificateurs[j - 1], j))
        self.ops = tuple(operations)                    # ops[0] = O_{T-1}, ..., ops[T-1] = O_0
        self.T = len(operations)                        # nombre de tours, sans le bourrage de l'article
        self.points = tuple(itertools.product(range(q), repeat=self.m))
        self._cache: dict = {}                          # memoire des valeurs P_t(sigma)

    def mle(self, sigma: tuple[int, ...]) -> int:
        """P_T en sigma : extension multilineaire de psi, evaluee dans F_q."""
        total = 0
        for booleens in itertools.product((0, 1), repeat=self.m):
            terme = 1 if self.psi(booleens) else 0      # valeur de psi sur l'affectation booleenne
            for xi, bi in zip(sigma, booleens):
                terme = terme * (xi if bi else 1 - xi) % self.q
            total = (total + terme) % self.q
        return total

    def link(self, kind: str, sj: int, g0: int, g1: int) -> int:
        """L_{t,alpha}(g0, g1) : a+b-ab (existe), ab (pour tout), interpolation (R_j)."""
        if kind == "E":
            return (g0 + g1 - g0 * g1) % self.q
        if kind == "A":
            return g0 * g1 % self.q
        return ((1 - sj) * g0 + sj * g1) % self.q

    def poly(self, t: int, sigma: tuple[int, ...]) -> int:
        """P_t(sigma) : P_T est l'extension multilineaire, puis P_t = O_t P_{t+1}."""
        cle = (t, sigma)
        if cle not in self._cache:
            if t == self.T:
                valeur = self.mle(sigma)
            else:
                kind, j = self.ops[self.T - 1 - t]
                g0 = self.poly(t + 1, sigma[: j - 1] + (0,) + sigma[j:])
                g1 = self.poly(t + 1, sigma[: j - 1] + (1,) + sigma[j:])
                valeur = self.link(kind, sigma[j - 1], g0, g1)
            self._cache[cle] = valeur
        return self._cache[cle]

    def restriction(self, t: int, sigma: tuple[int, ...]) -> tuple[int, int, int]:
        """Trois coefficients (a0, a1, a2) de g(X) = P_{t+1}(alpha[j <- X]), de degre au plus deux."""
        _kind, j = self.ops[self.T - 1 - t]
        v0, v1, v2 = (self.poly(t + 1, sigma[: j - 1] + (x,) + sigma[j:]) for x in (0, 1, 2))
        if self.q == 2:                                 # deux points seulement : representant affine
            return (v0, (v1 - v0) % self.q, 0)
        # a0 = g(0) ; a2 = seconde difference divisee (g(2) - 2 g(1) + g(0)) / 2 ; a1 = g(1) - g(0) - a2
        a2 = (v2 - 2 * v1 + v0) * pow(2, -1, self.q) % self.q
        return (v0, (v1 - v0 - a2) % self.q, a2)


def valeur_de_verite(inst: Instance) -> int:
    """I[Phi est vraie], par evaluation directe de la formule quantifiee sur les booleens."""

    def evaluer(affectation: tuple[int, ...], i: int) -> bool:
        if i == inst.m:
            return bool(inst.psi(affectation))
        sous_arbres = (evaluer(affectation + (x,), i + 1) for x in (0, 1))
        return any(sous_arbres) if inst.quantificateurs[i] == "E" else all(sous_arbres)

    return int(evaluer((), 0))


# Quatre formules : deux vraies, deux fausses, ecrites avec le meme psi deux fois deux
FORMULES = {
    "E x1 (x1)": (lambda b: b[0] == 1, ("E",)),
    "A x1 (x1)": (lambda b: b[0] == 1, ("A",)),
    "E x1 A x2 (x1 <-> x2)": (lambda b: b[0] == b[1], ("E", "A")),
    "A x1 E x2 (x1 <-> x2)": (lambda b: b[0] == b[1], ("A", "E")),
}

print("=== 1a. Quatre formules jouets sur F_5 : tours, operations, P_0, valeur de verite ===")
for nom, (psi, quant) in FORMULES.items():
    inst = Instance(5, psi, quant)
    p0 = {inst.poly(0, s) for s in inst.points}
    print(f"{nom:26s} T={inst.T} ops={inst.ops} P_0={p0} verite={valeur_de_verite(inst)}")

=== 1a. Quatre formules jouets sur F_5 : tours, operations, P_0, valeur de verite ===
E x1 (x1)                  T=1 ops=(('E', 1),) P_0={1} verite=1
A x1 (x1)                  T=1 ops=(('A', 1),) P_0={0} verite=0
E x1 A x2 (x1 <-> x2)      T=3 ops=(('R', 1), ('A', 2), ('E', 1)) P_0={0} verite=0
A x1 E x2 (x1 <-> x2)      T=3 ops=(('R', 1), ('E', 2), ('A', 1)) P_0={1} verite=1


### Lecture du résultat — quatre formules, une seule constante

Chaque ligne se lit ainsi : `T` est le nombre d'opérations avant d'atteindre la constante ; `ops` est la liste de ces opérations, dans l'ordre où on les applique à $P_T$ (`R j` : interpolation en $x_j$, c'est-à-dire $(1 - x_j)\,g(0) + x_j\,g(1)$ ; `E j` et `A j` : les deux quantificateurs) ; `P_0` est l'ensemble des valeurs prises par $P_0$ sur tout $\mathbb{F}_5^m$ ; `verite` est la valeur de vérité calculée **indépendamment**, par évaluation booléenne directe.

Deux lectures. D'abord $P_0$ est bien une **constante** — chaque ensemble est un singleton — et elle **coïncide** avec la valeur de vérité : c'est le Lemme 3.5 de l'article, cité, dont la dernière phrase est « The final polynomial is a constant equal to the truth value of the formula. » Ensuite l'ordre des opérations est celui que le même lemme prescrit ; à deux variables, il donne trois tours : interpoler en $x_1$, appliquer $Q_2$, appliquer $Q_1$. Pour ces quatre formules $T + 1$ vaut 2 ou 4, déjà une puissance de deux — la condition que l'article garantit par bourrage (Lemme 3.6).

In [4]:
# --- 1b. Degre individuel de chaque P_t en chaque variable (Lemme 3.5 : au plus deux) ---
def degre_individuel(valeurs: list[int], q: int) -> int:
    """Degre d'un polynome donne par ses q valeurs : differences finies successives modulo q."""
    cour = [v % q for v in valeurs]
    degre = 0
    while len(cour) > 1:
        cour = [(b - a) % q for a, b in zip(cour, cour[1:])]
        if not any(cour):
            break
        degre += 1
    return degre


def tableau_degres(inst: Instance) -> list[tuple[int, tuple[int, ...]]]:
    """Par tour t : degre individuel maximal de P_t en chaque variable, sur toutes les lignes de F_q^m."""
    tableau = []
    for t in range(inst.T + 1):
        par_variable = []
        for j in range(inst.m):
            pire = 0
            for autres in itertools.product(range(inst.q), repeat=inst.m - 1):
                ligne = [inst.poly(t, autres[:j] + (x,) + autres[j:]) for x in range(inst.q)]
                pire = max(pire, degre_individuel(ligne, inst.q))
            par_variable.append(pire)
        tableau.append((t, tuple(par_variable)))
    return tableau


print("=== 1b. Degre individuel de chaque P_t en chaque variable (q = 5) ===")
for nom, (psi, quant) in FORMULES.items():
    inst = Instance(5, psi, quant)
    tableau = tableau_degres(inst)
    print(f"{nom:26s} degres(t, par variable)={tableau} maximum={max(max(d) for _, d in tableau)}")

=== 1b. Degre individuel de chaque P_t en chaque variable (q = 5) ===
E x1 (x1)                  degres(t, par variable)=[(0, (0,)), (1, (1,))] maximum=1
A x1 (x1)                  degres(t, par variable)=[(0, (0,)), (1, (1,))] maximum=1
E x1 A x2 (x1 <-> x2)      degres(t, par variable)=[(0, (0, 0)), (1, (2, 0)), (2, (1, 1)), (3, (1, 1))] maximum=2
A x1 E x2 (x1 <-> x2)      degres(t, par variable)=[(0, (0, 0)), (1, (2, 0)), (2, (1, 1)), (3, (1, 1))] maximum=2


### Lecture du résultat — le degré monte à deux, et s'arrête là

Sur les quatre formules, le degré individuel maximal mesuré sur tous les $P_t$ et toutes les lignes de $\mathbb{F}_5^m$ est 2 : c'est la mesure, sur ce jouet, de la phrase du Lemme 3.5 « Every intermediate polynomial has individual degree at most two. » On voit **où** il monte. $P_T$, extension multilinéaire, est de degré 1 en chaque variable (les entrées `(1, 1)`) ; l'interpolation en $x_1$ ne change rien ; puis le quantificateur sur $x_2$ combine $g(0)$ et $g(1)$ par un **produit** — $g(0) g(1)$ pour « pour tout », $g(0) + g(1) - g(0) g(1)$ pour « il existe » — et le degré en $x_1$ passe à 2 (les entrées `(2, 0)`). Le dernier quantificateur élimine $x_1$ et laisse la constante `(0, 0)`.

Ce plafond de deux n'est pas un détail : c'est lui qui borne le nombre de racines d'un faux polynôme, donc la probabilité de tromper le contrôle, par $2/q$.

In [5]:
# --- 1c. Protocole 1 : qui parle, quand, et ce que le verificateur controle ---
Reponse = Callable[[int, tuple[int, ...], int], tuple[int, int, int]]


def deroule_protocole(inst: Instance, reponse_prouveur: Reponse,
                      defis: tuple[int, ...]) -> tuple[bool, list[tuple], list[tuple]]:
    """Protocole 1 pour un prouveur donne et une suite de defis : (accepte, trace, messages)."""
    q, T = inst.q, inst.T
    alpha, nu = (0,) * inst.m, 1                       # depart : le claim P_0(0...0) = 1
    trace, messages = [], []
    for t in range(T):
        a0, a1, a2 = reponse_prouveur(t, alpha, nu)    # message P -> V : les trois coefficients de g
        messages.append(("P -> V", (a0 % q, a1 % q, a2 % q)))
        kind, j = inst.ops[T - 1 - t]
        g0, g1 = a0 % q, (a0 + a1 + a2) % q            # g(0) et g(1) : les deux seules valeurs lues
        controle = inst.link(kind, alpha[j - 1], g0, g1)
        if controle != nu:                             # 2.b : le claim courant n'est pas respecte
            return False, trace + [(t, alpha, nu, (a0, a1, a2), controle, "rejet")], messages
        r = defis[t]                                   # message V -> P : defi uniforme sur F_q
        messages.append(("V -> P", r))
        trace.append((t, alpha, nu, (a0, a1, a2), controle, r))
        alpha = alpha[: j - 1] + (r,) + alpha[j:]
        nu = (a0 + a1 * r + a2 * r * r) % q            # message P -> V : g(r) devient le claim suivant
        messages.append(("P -> V", nu))
    accepte = nu == inst.poly(T, alpha)                # 3. : le claim final doit etre exact
    return accepte, trace + [(T, alpha, nu, None, inst.poly(T, alpha), "fin")], messages


def prouveur_honnete(inst: Instance) -> Reponse:
    """Le prouveur honnete : la vraie restriction g(X) = P_{t+1}(alpha[j <- X])."""
    return lambda t, alpha, _nu: inst.restriction(t, alpha)


print("=== 1c. Une execution du protocole, message par message ===")
inst = Instance(5, *FORMULES["A x1 E x2 (x1 <-> x2)"])
accepte, trace, messages = deroule_protocole(inst, prouveur_honnete(inst), (2, 3, 4))
print("trace honnete (formule vraie, m = 2, q = 5, defis 2, 3, 4) : accepte", accepte,
      "; messages =", len(messages))
for ligne in trace:
    print("   tour, alpha, claim, message g, controle, defi :", ligne)
print("   qui parle, dans l'ordre :", [qui for qui, _ in messages])
inst = Instance(5, *FORMULES["A x1 (x1)"])
accepte, trace, messages = deroule_protocole(inst, prouveur_honnete(inst), (3,))
print("formule fausse, m = 1, q = 5, defi 3 : accepte", accepte, "; messages =", messages)

=== 1c. Une execution du protocole, message par message ===
trace honnete (formule vraie, m = 2, q = 5, defis 2, 3, 4) : accepte True ; messages = 9
   tour, alpha, claim, message g, controle, defi : (0, (0, 0), 1, (1, 4, 1), 1, 2)
   tour, alpha, claim, message g, controle, defi : (1, (2, 0), 3, (4, 3, 0), 3, 3)
   tour, alpha, claim, message g, controle, defi : (2, (2, 3), 3, (3, 0, 0), 3, 4)
   tour, alpha, claim, message g, controle, defi : (3, (4, 3), 3, None, 3, 'fin')
   qui parle, dans l'ordre : ['P -> V', 'V -> P', 'P -> V', 'P -> V', 'V -> P', 'P -> V', 'P -> V', 'V -> P', 'P -> V']
formule fausse, m = 1, q = 5, defi 3 : accepte False ; messages = [('P -> V', (0, 1, 0))]


### Lecture du résultat — neuf messages pour trois tours, un seul pour un refus

Sur la formule vraie $\forall x_1 \exists x_2\,(x_1 \leftrightarrow x_2)$, à $q = 5$, avec les défis $(2, 3, 4)$, l'exécution honnête est acceptée. La trace se lit tour par tour. Au tour 0, le claim de départ est $P_0 = 1$ — le vérificateur affirme que la formule est vraie — et le prouveur envoie $g = (1, 4, 1)$, soit $g(X) = 1 + 4X + X^2$, qui est bien $P_1(X) = 1 - X + X^2$ dans $\mathbb{F}_5$ ; le contrôle « il existe » calcule $g(0) + g(1) - g(0)\,g(1) = 1 + 1 - 1 = 1 = \nu$ ; le défi $r = 2$ devient $x_1$, et $g(2) = 3$ devient le claim suivant. Au tour 1, le claim 3 est contrôlé par le produit de $\forall x_2$ : $g(0)\,g(1) = 4 \times 2 = 3$ ; défi $r = 3$, nouveau claim 3. Au tour 2, l'interpolation avec $x_1 = 2$ redonne 3 ; défi $r = 4$. Le dernier contrôle n'attend plus de message : le vérificateur compare le claim final à $P_3(4, 3) = 3$ et accepte.

Les messages alternent trois fois par tour — `P -> V` (les coefficients de $g$), `V -> P` (le défi), `P -> V` ($g(r)$) — d'où neuf messages pour trois tours. C'est **cette** structure que le §3 compressera en deux messages.

La formule fausse montre l'autre bout : à $m = 1$ et $q = 5$, le prouveur honnête envoie $g = (0, 1, 0)$, c'est-à-dire $g(X) = X = P_1(X)$, et le contrôle « pour tout » calcule $g(0)\,g(1) = 0 \ne 1$ : le protocole s'arrête **après un seul message**, sans qu'aucun défi n'ait été tiré. Un prouveur honnête sur une formule fausse n'a rien à annoncer qui passe le premier contrôle.

In [6]:
# --- 1d. Le prouveur honnete sur TOUTES les suites de defis, et rien que lui ---
print("=== 1d. prouveur honnete : toutes les suites de defis ===")
for nom, (psi, quant) in FORMULES.items():
    for q in (2, 3, 5, 7):
        inst = Instance(q, psi, quant)
        resultats = [deroule_protocole(inst, prouveur_honnete(inst), defis)[0]
                     for defis in itertools.product(range(q), repeat=inst.T)]
        print(f"{nom:26s} q={q} suites={len(resultats):4d} acceptees={sum(resultats):4d}")

=== 1d. prouveur honnete : toutes les suites de defis ===
E x1 (x1)                  q=2 suites=   2 acceptees=   2
E x1 (x1)                  q=3 suites=   3 acceptees=   3
E x1 (x1)                  q=5 suites=   5 acceptees=   5
E x1 (x1)                  q=7 suites=   7 acceptees=   7
A x1 (x1)                  q=2 suites=   2 acceptees=   0
A x1 (x1)                  q=3 suites=   3 acceptees=   0
A x1 (x1)                  q=5 suites=   5 acceptees=   0
A x1 (x1)                  q=7 suites=   7 acceptees=   0
E x1 A x2 (x1 <-> x2)      q=2 suites=   8 acceptees=   0
E x1 A x2 (x1 <-> x2)      q=3 suites=  27 acceptees=   0
E x1 A x2 (x1 <-> x2)      q=5 suites= 125 acceptees=   0
E x1 A x2 (x1 <-> x2)      q=7 suites= 343 acceptees=   0
A x1 E x2 (x1 <-> x2)      q=2 suites=   8 acceptees=   8
A x1 E x2 (x1 <-> x2)      q=3 suites=  27 acceptees=  27
A x1 E x2 (x1 <-> x2)      q=5 suites= 125 acceptees= 125
A x1 E x2 (x1 <-> x2)      q=7 suites= 343 acceptees= 343


### Lecture du résultat — complétude parfaite, et ce qu'elle ne dit pas

Sur les formules **vraies**, toutes les suites de défis sont acceptées, à tous les $q$ : $q^T$ suites, $q^T$ acceptations — une complétude de 1, sans erreur. Sur les formules **fausses**, zéro, à tous les $q$ : le prouveur honnête y est rejeté dès le premier contrôle, comme au §1c.

Ce tableau ne teste que la **complétude**. La question de la **solidité** — un prouveur malhonnête peut-il faire accepter une formule fausse, et à quelle fréquence — n'est pas une question sur le prouveur honnête : c'est un problème de maximisation sur tous ses messages possibles, que la cellule 1e résout exactement.

In [7]:
# --- 1e. Le meilleur prouveur trompeur, par programme dynamique exact ---
def meilleure_triche(inst: Instance) -> float:
    """Probabilite exacte d'acceptation du meilleur prouveur trompeur (programme dynamique).

    V[t][alpha, nu] : meilleur espoir d'acceptation depuis le tour t, le point alpha et le claim nu.
    Le prouveur choisit trois coefficients ; le defi r est uniforme sur F_q.
    """
    q, T = inst.q, inst.T
    sigmas = inst.points
    n_s = len(sigmas)
    rang = {s: i for i, s in enumerate(sigmas)}
    coefficients = NP.array(list(itertools.product(range(q), repeat=3)), dtype=NP.int64)
    defis = NP.arange(q)
    valeurs = (coefficients[:, [0]] + coefficients[:, [1]] * defis
               + coefficients[:, [2]] * defis ** 2) % q
    g0, g1 = valeurs[:, 0], valeurs[:, 1]
    lignes = NP.arange(n_s)
    V = NP.zeros((n_s, q))
    for i, s in enumerate(sigmas):                     # tour T : accepte si le claim est correct
        V[i, inst.poly(T, s)] = 1.0
    for t in range(T - 1, -1, -1):
        kind, j = inst.ops[T - 1 - t]
        sj = NP.array([s[j - 1] for s in sigmas])
        suiv = NP.empty((n_s, q), dtype=NP.int64)      # alpha[j <- x] pour chaque point et chaque x
        for i, s in enumerate(sigmas):
            for x in range(q):
                suiv[i, x] = rang[s[: j - 1] + (x,) + s[j:]]
        Vn = NP.zeros((n_s, q))
        for k in range(len(coefficients)):             # un message g a la fois
            if kind == "E":
                lien = (g0[k] + g1[k] - g0[k] * g1[k]) % q
            elif kind == "A":
                lien = g0[k] * g1[k] % q
            else:
                lien = ((1 - sj) * g0[k] + sj * g1[k]) % q
            espoir = V[suiv, valeurs[k]].mean(axis=1)  # moyenne sur le defi r uniforme
            NP.maximum.at(Vn, (lignes, lien % q), espoir)   # ce message ne passe que pour nu = L(g)
        V = Vn
    return float(V[rang[(0,) * inst.m], 1])


print("=== 1e. meilleure triche : probabilite d'acceptation exacte du meilleur prouveur trompeur ===")
for q in (2, 3, 5, 7, 11, 13):
    for nom in ("A x1 (x1)", "E x1 (x1)"):
        inst = Instance(q, *FORMULES[nom])
        p = meilleure_triche(inst)
        print(f"{nom:14s} q={q:2d} T={inst.T} P(accept)={p:.6f}  2T/q={2 * inst.T / q:.6f}")
for q in (3, 5, 7, 11, 13):
    for nom in ("E x1 A x2 (x1 <-> x2)", "A x1 E x2 (x1 <-> x2)"):
        inst = Instance(q, *FORMULES[nom])
        p = meilleure_triche(inst)
        print(f"{nom:24s} q={q:2d} T={inst.T} P(accept)={p:.6f}  2T/q={2 * inst.T / q:.6f} "
              f"ratio={p / (2 * inst.T / q):.4f}")
for q in (2, 3, 5, 7, 11, 13):
    inst = Instance(q, *FORMULES["A x1 (x1)"])
    gain = sum(deroule_protocole(inst, lambda t, a, n: (1, 0, 0), (r,))[0] for r in range(q))
    print(f"tricheur g = 1 sur A x1 (x1), q={q} : {gain}/{q} = {gain / q:.6f}")

=== 1e. meilleure triche : probabilite d'acceptation exacte du meilleur prouveur trompeur ===
A x1 (x1)      q= 2 T=1 P(accept)=0.500000  2T/q=1.000000
E x1 (x1)      q= 2 T=1 P(accept)=1.000000  2T/q=1.000000
A x1 (x1)      q= 3 T=1 P(accept)=0.666667  2T/q=0.666667
E x1 (x1)      q= 3 T=1 P(accept)=1.000000  2T/q=0.666667
A x1 (x1)      q= 5 T=1 P(accept)=0.400000  2T/q=0.400000
E x1 (x1)      q= 5 T=1 P(accept)=1.000000  2T/q=0.400000
A x1 (x1)      q= 7 T=1 P(accept)=0.285714  2T/q=0.285714
E x1 (x1)      q= 7 T=1 P(accept)=1.000000  2T/q=0.285714
A x1 (x1)      q=11 T=1 P(accept)=0.181818  2T/q=0.181818
E x1 (x1)      q=11 T=1 P(accept)=1.000000  2T/q=0.181818
A x1 (x1)      q=13 T=1 P(accept)=0.153846  2T/q=0.153846
E x1 (x1)      q=13 T=1 P(accept)=1.000000  2T/q=0.153846


E x1 A x2 (x1 <-> x2)    q= 3 T=3 P(accept)=0.962963  2T/q=2.000000 ratio=0.4815
A x1 E x2 (x1 <-> x2)    q= 3 T=3 P(accept)=1.000000  2T/q=2.000000 ratio=0.5000
E x1 A x2 (x1 <-> x2)    q= 5 T=3 P(accept)=0.784000  2T/q=1.200000 ratio=0.6533
A x1 E x2 (x1 <-> x2)    q= 5 T=3 P(accept)=1.000000  2T/q=1.200000 ratio=0.8333
E x1 A x2 (x1 <-> x2)    q= 7 T=3 P(accept)=0.635569  2T/q=0.857143 ratio=0.7415
A x1 E x2 (x1 <-> x2)    q= 7 T=3 P(accept)=1.000000  2T/q=0.857143 ratio=1.1667


E x1 A x2 (x1 <-> x2)    q=11 T=3 P(accept)=0.452292  2T/q=0.545455 ratio=0.8292


A x1 E x2 (x1 <-> x2)    q=11 T=3 P(accept)=1.000000  2T/q=0.545455 ratio=1.8333


E x1 A x2 (x1 <-> x2)    q=13 T=3 P(accept)=0.394174  2T/q=0.461538 ratio=0.8540


A x1 E x2 (x1 <-> x2)    q=13 T=3 P(accept)=1.000000  2T/q=0.461538 ratio=2.1667
tricheur g = 1 sur A x1 (x1), q=2 : 1/2 = 0.500000
tricheur g = 1 sur A x1 (x1), q=3 : 1/3 = 0.333333
tricheur g = 1 sur A x1 (x1), q=5 : 1/5 = 0.200000
tricheur g = 1 sur A x1 (x1), q=7 : 1/7 = 0.142857
tricheur g = 1 sur A x1 (x1), q=11 : 1/11 = 0.090909
tricheur g = 1 sur A x1 (x1), q=13 : 1/13 = 0.076923


### Lecture du résultat — la probabilité exacte du meilleur menteur

Le programme dynamique ne simule aucun tirage : à chaque tour il énumère les $q^3$ messages possibles, garde ceux que le contrôle accepte et prend le maximum, en moyennant sur les défis uniformes. Les nombres sont donc **exacts**, pas estimés — un sondage Monte-Carlo aurait donné une erreur d'échantillonnage là où ici il n'y en a pas.

Trois régimes se lisent. (1) Sur la formule fausse à une variable ($T = 1$), la probabilité vaut $2/q$ pour $q \ge 3$ : exactement la borne de l'article, atteinte. À $q = 2$ elle vaut $1/2$, sous la borne $2T/q = 1$ — qui, valant 1, ne contraint rien. (2) Le menteur naïf qui envoie $g \equiv 1$ passe toujours le contrôle mais n'est cru que si le défi tombe sur $r = 1$ : $1/q$, deux fois moins que le menteur optimal dès que $q \ge 3$ (à $q = 2$, les deux valent $1/2$). (3) Sur la formule fausse à trois tours, la borne $2T/q$ n'est **informative** que si $q > 2T$ : à $q = 3$ et $q = 5$ elle dépasse 1, et la valeur mesurée est élevée (0,963 puis 0,784) ; à $q \ge 7$ elle mord (0,636 < 0,857 ; 0,452 < 0,545 ; 0,394 < 0,462). L'article choisit $q = 8(T+1)$, où l'union sur les tours donne $T/(4(T+1)) < 1/4$ : c'est ce choix de corps, autant que la structure du protocole, qui rend la solidité exploitable.

Les lignes des formules **vraies** valent 1 sans qu'il y ait triche : le meilleur prouveur y est simplement le prouveur honnête, et la complétude est parfaite. Un maximum sur toutes les stratégies inclut toujours l'honnête.

### Exercice 1 — Le meilleur menteur, message par message

La cellule 1e a obtenu la probabilité du meilleur menteur par programmation dynamique : sur l'instance fausse `A x1 (x1)` à $q = 5$, elle vaut $2/5$. Reprenez le même objet par énumération brute, sans récurrence.

1. Un message du premier tour est un triplet $(a_0, a_1, a_2)$ de $\mathbb{F}_5$ : il y en a $125$. Gardez ceux qui passent le contrôle du premier tour.
2. Pour chaque message gardé, faites jouer un prouveur qui renvoie **toujours le même message**. Il essuie $q = 5$ défis, un par valeur de $r$. Comptez les défis qui mènent à l'acceptation.
3. Divisez le meilleur compte par $5$ et comparez à `meilleure_triche(inst_ex1)`.

**Ce qu'il faut regarder.** Combien de messages survivent au contrôle ? Parmi eux, combien échouent sur les cinq défis ? Le maximum est-il celui que la programmation dynamique avait trouvé ?

In [8]:
# Exercice 1 : le meilleur menteur a q = 5, message par message
inst_ex1 = Instance(5, *FORMULES["A x1 (x1)"])           # instance fausse : A x1, psi(x1) = x1
messages_possibles = list(itertools.product(range(5), repeat=3))
messages_admis = []                                       # TODO etudiant
# Indice : au premier tour, le controle compare inst_ex1.link("A", 0, g0, g1) a nu = 1,
# avec g0 = a0 % 5 et g1 = (a0 + a1 + a2) % 5. Les messages admis sont ceux pour qui cette egalite a lieu.
# Etape 1 : garder les messages qui passent ce controle.
# Etape 2 : pour chaque message admis, compter les defis r en 0..4 acceptes par deroule_protocole
#           (le prouveur renvoie toujours le meme message : voir le parametre reponse_prouveur).
# Etape 3 : comparer le meilleur compte, divise par 5, a meilleure_triche(inst_ex1).
print("Exercice 1 a completer : messages testes =", len(messages_possibles),
      "; messages admis =", len(messages_admis))

Exercice 1 a completer : messages testes = 125 ; messages admis = 0


## 2. Où la transformation vers `CIUniform` intervient : l'échantillonneur

Le §1 a rejoué le sum-check sous sa forme d'origine : $T$ tours, trois messages par tour. L'article ne le conserve pas sous cette forme. Il le transforme en une instance d'un problème à promesse, **Close Image to Uniform** (`CIUniform`). L'abstract le dit en une phrase :

> « Our main technical contributions show that Close Image to Uniform (CIUniform), which is naturally in the stoquastic version of qq-QAM, is PSPACE-hard via transforming the sum-check protocol of Shamir (JACM, 1992) and Shen (JACM, 1992) into a CIUniform instance consisting of an efficient sampler whose image distribution is exactly uniform for some input distribution when the underlying quantified Boolean formula is true. »

**Définition 3.1 (citée).** « Let $S$ be a polynomial-size Boolean circuit that takes an $m_{in}$-bit string $u$ and an $m_+$-bit random seed $r$ as input and outputs an $m_{out}$-bit string $u'$. Equivalently, for each $u \in \{0,1\}^{m_{in}}$, $S$ serves as a sampler that draws $r \sim U_{m_+}$ independently and outputs $S(u, r)$. » La matrice de transition est $M_S(u', u) = 2^{-m_+}\,\bigl|\{\,r : S(u, r) = u'\,\}\bigr|$, et `CIUniform`$[\eta]$ demande de distinguer :

- **oui** — il existe une loi d'entrée $p$ dont l'image $M_S p$ est **exactement uniforme** ;
- **non** — toute loi d'entrée $p$ donne une image à distance **au moins $\eta$** (en variation totale) de l'uniforme.

**La transformation intervient dans l'échantillonneur $S$.** Les $T$ tours du §1c — chacun avec son contrôle et son défi — deviennent **une seule application** d'un circuit à graine. L'entrée est $u = (t, \alpha, g)$ : le tour, le point d'évaluation et le polynôme annoncé. La graine $(r, v)$ tient le rôle du vérificateur : $r$ est le défi, $v$ un faux claim. L'équation (3.4) de l'article définit

$$S(u; r, v) = \begin{cases} C_{next}(u, r) & \text{si }(t, \alpha, v) = C_{now}(u),\\ (t, \alpha, v) & \text{sinon},\end{cases}$$

avec, pour $t < T$ :

$$C_{now}(u) = \bigl(t, \alpha, L_{t,\alpha}(g(0), g(1))\bigr), \qquad C_{next}(u, r) = \bigl(t + 1, \alpha[j_t \leftarrow r], g(r)\bigr),$$

et, pour $t = T$ : $C_{now}(u) = (T, \alpha, P_T(\alpha))$ et $C_{next}(u, r) = (0, \alpha, 1)$. C'est la logique du §1c, avalée dans un circuit réversible : $C_{now}$ est le **contrôle** — le claim que le message $g$ défend —, $C_{next}$ est le **tour suivant** — défi $r$, nouveau claim $g(r)$. Le lien avec le §1b est direct : le contrôle ne lit que $g(0)$ et $g(1)$ d'un polynôme de degré au plus deux, ce qui garde le circuit petit.

Deux transpositions, écrites pour éviter tout contresens. (i) L'article travaille sur $\mathbb{F}_q$ avec $q = 8(T+1)$, une puissance de deux, et compte ses graines en bits ; ici $q$ est premier et petit, la graine est un couple $(r, v) \in \mathbb{F}_q^2$ et chacune des $q^2$ graines a le poids $1/q^2$. (ii) **La PSPACE-difficulté de `CIUniform` est citée, pas rejouée** : le carnet vérifie deux lemmes de la section 3 de l'article sur des jouets — le lemme 3.7 (complétude : la loi honnête donne une image uniforme si la formule est vraie) et le lemme 3.8 (solidité : toute loi est éloignée si elle est fausse) —, il ne montre pas que `CIUniform` est PSPACE-difficile.

In [9]:
# --- 2a. L'echantillonneur S : entree (t, alpha, g), graine (r, v), sortie (t', alpha', nu) ---
Point = tuple[int, ...]
Entree = tuple[int, Point, tuple[int, int, int]]
Sortie = tuple[int, Point, int]


class Echantillonneur:
    """Le circuit S de la section 3.2 : entree (t, alpha, g) + graine (r, v) -> sortie (t', alpha', nu)."""

    def __init__(self, inst: Instance) -> None:
        self.inst = inst
        q, T = inst.q, inst.T
        self.entrees = tuple((t, s, g) for t in range(T + 1) for s in inst.points
                             for g in itertools.product(range(q), repeat=3))
        self.sorties = tuple((t, s, v) for t in range(T + 1) for s in inst.points for v in range(q))
        self.rang = {u: i for i, u in enumerate(self.sorties)}
        self.nb_graines = q * q                       # une graine = un couple (r, v), d'indice q r + v
        self._table: NP.ndarray | None = None
        self.matrice = self._matrice()

    def valeur(self, g: tuple[int, int, int], r: int) -> int:
        """g(r) pour un polynome donne par ses trois coefficients."""
        return (g[0] + g[1] * r + g[2] * r * r) % self.inst.q

    def claim_courant(self, u: Entree) -> int:
        """Valeur du claim C_now(u), equation (3.3a) : celle pour laquelle g passe le controle."""
        t, s, g = u
        if t == self.inst.T:
            return self.inst.poly(t, s)
        kind, j = self.inst.ops[self.inst.T - 1 - t]
        return self.inst.link(kind, s[j - 1], g[0], self.valeur(g, 1))

    def claim_suivant(self, u: Entree, r: int) -> Sortie:
        """C_next(u, r), equation (3.3b) : tour t+1 au point alpha[j <- r], ou retour au tour 0."""
        t, s, g = u
        if t == self.inst.T:
            return (0, s, 1)
        _kind, j = self.inst.ops[self.inst.T - 1 - t]
        return (t + 1, s[: j - 1] + (r,) + s[j:], self.valeur(g, r))

    def echantillonne(self, u: Entree, r: int, v: int) -> Sortie:
        """S(u ; r, v), equation (3.4) : la graine v ne remplace que si elle tombe sur C_now(u)."""
        if v == self.claim_courant(u):
            return self.claim_suivant(u, r)
        return (u[0], u[1], v)

    def table(self) -> NP.ndarray:
        """Matrice (entrees x graines) des indices de sortie de S(u ; graine)."""
        if self._table is None:
            q = self.inst.q
            self._table = NP.array([[self.rang[self.echantillonne(u, s // q, s % q)]
                                    for s in range(self.nb_graines)] for u in self.entrees], dtype=NP.int64)
        return self._table

    def _matrice(self) -> csr_matrix:
        """M_S : matrice de transition (sorties x entrees), chaque graine de poids 1/q^2."""
        table = self.table()
        lignes = table.ravel()                           # une entree par (entree, graine), sommee par csr_matrix
        colonnes = NP.repeat(NP.arange(len(self.entrees)), self.nb_graines)
        poids = NP.full(lignes.shape, 1.0 / self.nb_graines)
        return csr_matrix((poids, (lignes, colonnes)), shape=(len(self.sorties), len(self.entrees)))

    def p_honnete(self) -> NP.ndarray:
        """La distribution p du Lemme 3.7 : (t, alpha) uniforme, g = g* puis 0 au dernier tour."""
        p = NP.zeros(len(self.entrees))
        for t in range(self.inst.T + 1):
            for s in self.inst.points:
                g = (0, 0, 0) if t == self.inst.T else self.inst.restriction(t, s)
                p[self.rang_entree(t, s, g)] = 1.0 / ((self.inst.T + 1) * len(self.inst.points))
        return p

    def rang_entree(self, t: int, s: Point, g: tuple[int, int, int]) -> int:
        """Indice de l'entree (t, s, g) dans la liste `entrees`."""
        q = self.inst.q
        interne = g[0] * q * q + g[1] * q + g[2]
        return (t * len(self.inst.points) + self.inst.points.index(s)) * q ** 3 + interne

    def equation_3_5(self, p: NP.ndarray) -> NP.ndarray:
        """Membre de droite de (3.5) : Pr[(t, alpha, v)] + (1/q)(Pr[C_next] - Pr[C_now])."""
        q = self.inst.q
        tirage = NP.zeros(len(self.sorties))
        suivant = NP.zeros(len(self.sorties))
        courant = NP.zeros(len(self.sorties))
        for i, u in enumerate(self.entrees):
            for v in range(q):                          # (t, alpha, v) : v uniforme
                tirage[self.rang[(u[0], u[1], v)]] += p[i] / q
            courant[self.rang[(u[0], u[1], self.claim_courant(u))]] += p[i]
            for r in range(q):                          # r uniforme
                suivant[self.rang[self.claim_suivant(u, r)]] += p[i] / q
        return tirage + (suivant - courant) / q

    def tv(self, p: NP.ndarray) -> float:
        """Distance en variation totale entre M_S p et la loi uniforme sur les sorties."""
        return float(0.5 * NP.abs(self.matrice @ p - 1.0 / len(self.sorties)).sum())

    def tv_minimal(self) -> tuple[float, NP.ndarray]:
        """min_p TV(M_S p, U) par programme lineaire : y >= M p - U et y >= U - M p."""
        M = self.matrice
        n_out, n_in = M.shape
        uniforme = NP.full(n_out, 1.0 / n_out)
        identite = identity(n_out, format="csr")
        A_ub = vstack([hstack([M, -identite]), hstack([-M, -identite])], format="csr")
        b_ub = NP.concatenate([uniforme, -uniforme])
        A_eq = csr_matrix((NP.ones(n_in), (NP.zeros(n_in), NP.arange(n_in))), shape=(1, n_in + n_out))
        objectif = NP.concatenate([NP.zeros(n_in), NP.ones(n_out)])
        resultat = linprog(objectif, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=[1.0],
                           bounds=(0, None), method="highs")
        return float(resultat.fun) / 2.0, resultat.x[:n_in]


print("=== 2a. une passe de l'echantillonneur ===")
inst = Instance(5, *FORMULES["A x1 (x1)"])
ech = Echantillonneur(inst)
u = (0, (0,), (1, 0, 0))                                # claim trompeur : g = 1, alors que P_1 = x1
print("instance fausse A x1 (x1), q = 5 ; entree u = (t, alpha, g) =", u, "; C_now(u) =", ech.claim_courant(u))
for r in range(5):
    print("   r =", r, ":", [ech.echantillonne(u, r, v) for v in range(5)])

=== 2a. une passe de l'echantillonneur ===
instance fausse A x1 (x1), q = 5 ; entree u = (t, alpha, g) = (0, (0,), (1, 0, 0)) ; C_now(u) = 1
   r = 0 : [(0, (0,), 0), (1, (0,), 1), (0, (0,), 2), (0, (0,), 3), (0, (0,), 4)]
   r = 1 : [(0, (0,), 0), (1, (1,), 1), (0, (0,), 2), (0, (0,), 3), (0, (0,), 4)]
   r = 2 : [(0, (0,), 0), (1, (2,), 1), (0, (0,), 2), (0, (0,), 3), (0, (0,), 4)]
   r = 3 : [(0, (0,), 0), (1, (3,), 1), (0, (0,), 2), (0, (0,), 3), (0, (0,), 4)]
   r = 4 : [(0, (0,), 0), (1, (4,), 1), (0, (0,), 2), (0, (0,), 3), (0, (0,), 4)]


### Lecture du résultat — la graine tire le contrôle du vérificateur

Sur la formule fausse $\forall x_1\,(x_1)$, à $q = 5$, le menteur naïf du §1e envoie $g = (1, 0, 0)$, c'est-à-dire $g \equiv 1$, alors que le vrai polynôme est $P_1(X) = X$. Ce message défend le claim $1$ : $C_{now}(u) = 1$.

Chaque ligne fixe le défi $r$ et parcourt les cinq valeurs de $v$. Une seule graine par ligne fait avancer le protocole : $(r, 1)$, celle dont le second terme tombe **exactement** sur le claim défendu, et la sortie est alors le tour suivant, $(1, (r,), 1)$. Pour les quatre autres graines, la sortie est $(0, (0,), v)$ : l'échantillonneur rend le couple (tour, point) inchangé et y inscrit la valeur $v$.

Le circuit ne « sait » pas si le claim est vrai. Il imite le vérificateur en tirant $v$ au hasard : le message passe au tour suivant seulement si ce tirage tombe sur le claim qu'il défend, avec la probabilité $1/q$. C'est tout le contenu de l'équation (3.5), que la cellule suivante vérifie.

In [10]:
# --- 2b. L'equation (3.5) sur des lois quelconques, et la loi honnete du Lemme 3.7 ---
print("=== 2b. equation (3.5) et Lemme 3.7 ===")
generateur_lois = NP.random.default_rng(SEED)           # flux dedie aux lois aleatoires de la section 2
for (m, q) in ((1, 2), (1, 3), (1, 5), (1, 7), (2, 3), (2, 5)):
    for nom, (psi, quant) in FORMULES.items():
        if len(quant) != m:
            continue
        inst = Instance(q, psi, quant)
        ech = Echantillonneur(inst)
        p = ech.p_honnete()
        ecart = max([float(NP.abs(ech.matrice @ p - ech.equation_3_5(p)).max())] +
                    [float(NP.abs(ech.matrice @ alea - ech.equation_3_5(alea)).max())
                     for alea in generateur_lois.dirichlet(NP.ones(len(ech.entrees)), size=3)])
        colonnes = sorted({round(float(c), 9) for c in NP.asarray(ech.matrice.sum(axis=0)).ravel()})
        print(f"m={m} q={q} {nom:24s} entrees={len(ech.entrees):5d} sorties={len(ech.sorties):4d} "
              f"sommes des colonnes={colonnes} (3.5) max ec={ecart:.1e} TV honnete={ech.tv(p):.6f} "
              f"attendu={(1 - valeur_de_verite(inst)) / (q * (inst.T + 1)):.6f}")

=== 2b. equation (3.5) et Lemme 3.7 ===
m=1 q=2 E x1 (x1)                entrees=   32 sorties=   8 sommes des colonnes=[1.0] (3.5) max ec=2.8e-17 TV honnete=0.000000 attendu=0.000000
m=1 q=2 A x1 (x1)                entrees=   32 sorties=   8 sommes des colonnes=[1.0] (3.5) max ec=5.6e-17 TV honnete=0.250000 attendu=0.250000
m=1 q=3 E x1 (x1)                entrees=  162 sorties=  18 sommes des colonnes=[1.0] (3.5) max ec=5.6e-17 TV honnete=0.000000 attendu=0.000000
m=1 q=3 A x1 (x1)                entrees=  162 sorties=  18 sommes des colonnes=[1.0] (3.5) max ec=5.6e-17 TV honnete=0.166667 attendu=0.166667
m=1 q=5 E x1 (x1)                entrees= 1250 sorties=  50 sommes des colonnes=[1.0] (3.5) max ec=7.6e-17 TV honnete=0.000000 attendu=0.000000
m=1 q=5 A x1 (x1)                entrees= 1250 sorties=  50 sommes des colonnes=[1.0] (3.5) max ec=9.7e-17 TV honnete=0.100000 attendu=0.100000


m=1 q=7 E x1 (x1)                entrees= 4802 sorties=  98 sommes des colonnes=[1.0] (3.5) max ec=1.0e-16 TV honnete=0.000000 attendu=0.000000


m=1 q=7 A x1 (x1)                entrees= 4802 sorties=  98 sommes des colonnes=[1.0] (3.5) max ec=1.0e-16 TV honnete=0.071429 attendu=0.071429
m=2 q=3 E x1 A x2 (x1 <-> x2)    entrees=  972 sorties= 108 sommes des colonnes=[1.0] (3.5) max ec=8.7e-18 TV honnete=0.083333 attendu=0.083333
m=2 q=3 A x1 E x2 (x1 <-> x2)    entrees=  972 sorties= 108 sommes des colonnes=[1.0] (3.5) max ec=1.4e-17 TV honnete=0.000000 attendu=0.000000


m=2 q=5 E x1 A x2 (x1 <-> x2)    entrees=12500 sorties= 500 sommes des colonnes=[1.0] (3.5) max ec=9.1e-18 TV honnete=0.050000 attendu=0.050000


m=2 q=5 A x1 E x2 (x1 <-> x2)    entrees=12500 sorties= 500 sommes des colonnes=[1.0] (3.5) max ec=9.5e-18 TV honnete=0.000000 attendu=0.000000


### Lecture du résultat — une identité exacte, et une distance $1/(q(T+1))$

Trois choses se lisent, toutes à la précision flottante près.

**Les sommes des colonnes valent 1.** Pour chaque entrée, les $q^2$ graines se répartissent sur les sorties sans en perdre : $M_S$ est une matrice stochastique, chaque colonne est une loi.

**L'équation (3.5) est une identité — sur la loi honnête comme sur trois lois tirées au hasard.** L'article écrit, pour toute partie $E$ de sorties,

$$\Pr[u' \in E] = \Pr[(t, \alpha, v) \in E] + \frac{1}{q}\bigl(\Pr[C_{next}(u, r) \in E] - \Pr[C_{now}(u) \in E]\bigr).$$

L'image est donc « l'entrée, avec $v$ en dernière composante », corrigée d'un terme de poids $1/q$ qui déplace la masse du claim défendu $C_{now}(u)$ vers le tour suivant $C_{next}(u, r)$. C'est ce que montrait la cellule 2a : une graine sur $q$ fait avancer le protocole. L'écart mesuré est de l'ordre de $10^{-16}$ sur les douze instances.

**La loi honnête donne l'image annoncée par le lemme 3.7.** « If Φ is true, then $M_S p = U_{m_{out}}$. If Φ is false, then $TV(M_S p, U_{m_{out}}) = 1/(q(T + 1))$. » Mesuré : distance nulle sur les six instances vraies, et sur les fausses 0,25 ; 0,166 667 ; 0,1 ; 0,071 429 ($T = 1$) puis 0,083 333 et 0,05 ($T = 3$) — à chaque fois la valeur $1/(q(T+1))$ de la dernière colonne. **Formule vraie, l'image est exactement uniforme : c'est le cas « oui » de `CIUniform`.**

Il reste l'autre moitié de la promesse. Sur une formule fausse, la loi honnête est éloignée de l'uniforme — mais le cas « non » exige que **toute** loi d'entrée le soit. C'est l'objet du lemme 3.8, et de la cellule suivante.

In [11]:
# --- 2c. Lemme 3.8 : distance minimale sur TOUTES les lois d'entree, par programme lineaire ---
print("=== 2c. Lemme 3.8 : distance minimale sur toutes les lois d'entree ===")
for (m, q) in ((1, 2), (1, 3), (1, 5), (1, 7), (1, 11), (2, 2), (2, 3), (2, 5)):
    for nom, (psi, quant) in FORMULES.items():
        if len(quant) != m:
            continue
        inst = Instance(q, psi, quant)
        ech = Echantillonneur(inst)
        tv_min, _p_min = ech.tv_minimal()
        borne = 1 / (4 * q * (inst.T + 1))
        intermediaire = (1 / (q + 1)) * (1 / (inst.T + 1) - 3 / q)   # quantite du fil du lemme 3.8
        vraie = bool(valeur_de_verite(inst))
        verdict = "vraie" if vraie else ("OK" if tv_min >= borne - 1e-9 else "ECHEC")
        print(f"m={m} q={q:2d} {nom:24s} verite={int(vraie)} min TV={tv_min:.6f} "
              f"borne 1/(4q(T+1))={borne:.6f} {verdict} q>3(T+1)={q > 3 * (inst.T + 1)} "
              f"intermediaire={intermediaire:+.6f}")

inst = Instance(2, *FORMULES["A x1 (x1)"])
ech = Echantillonneur(inst)
tv_min, p_min = ech.tv_minimal()
print("loi d'entree qui rend l'image exactement uniforme (instance fausse, q = 2) :")
for i, u in enumerate(ech.entrees):
    if p_min[i] > 1e-9:
        print("   ", u, f"p = {p_min[i]:.6f}")
print("image M_S p :", [round(float(x), 6) for x in ech.matrice @ p_min],
      " uniforme =", 1 / len(ech.sorties))

=== 2c. Lemme 3.8 : distance minimale sur toutes les lois d'entree ===
m=1 q= 2 E x1 (x1)                verite=1 min TV=0.000000 borne 1/(4q(T+1))=0.062500 vraie q>3(T+1)=False intermediaire=-0.333333
m=1 q= 2 A x1 (x1)                verite=0 min TV=0.000000 borne 1/(4q(T+1))=0.062500 ECHEC q>3(T+1)=False intermediaire=-0.333333
m=1 q= 3 E x1 (x1)                verite=1 min TV=0.000000 borne 1/(4q(T+1))=0.041667 vraie q>3(T+1)=False intermediaire=-0.125000
m=1 q= 3 A x1 (x1)                verite=0 min TV=0.047619 borne 1/(4q(T+1))=0.041667 OK q>3(T+1)=False intermediaire=-0.125000
m=1 q= 5 E x1 (x1)                verite=1 min TV=0.000000 borne 1/(4q(T+1))=0.025000 vraie q>3(T+1)=False intermediaire=-0.016667
m=1 q= 5 A x1 (x1)                verite=0 min TV=0.047000 borne 1/(4q(T+1))=0.025000 OK q>3(T+1)=False intermediaire=-0.016667


m=1 q= 7 E x1 (x1)                verite=1 min TV=0.000000 borne 1/(4q(T+1))=0.017857 vraie q>3(T+1)=True intermediaire=+0.008929


m=1 q= 7 A x1 (x1)                verite=0 min TV=0.043064 borne 1/(4q(T+1))=0.017857 OK q>3(T+1)=True intermediaire=+0.008929


m=1 q=11 E x1 (x1)                verite=1 min TV=0.000000 borne 1/(4q(T+1))=0.011364 vraie q>3(T+1)=True intermediaire=+0.018939


m=1 q=11 A x1 (x1)                verite=0 min TV=0.033538 borne 1/(4q(T+1))=0.011364 OK q>3(T+1)=True intermediaire=+0.018939
m=2 q= 2 E x1 A x2 (x1 <-> x2)    verite=0 min TV=0.000000 borne 1/(4q(T+1))=0.031250 ECHEC q>3(T+1)=False intermediaire=-0.416667
m=2 q= 2 A x1 E x2 (x1 <-> x2)    verite=1 min TV=0.000000 borne 1/(4q(T+1))=0.031250 vraie q>3(T+1)=False intermediaire=-0.416667
m=2 q= 3 E x1 A x2 (x1 <-> x2)    verite=0 min TV=0.003086 borne 1/(4q(T+1))=0.020833 ECHEC q>3(T+1)=False intermediaire=-0.187500
m=2 q= 3 A x1 E x2 (x1 <-> x2)    verite=1 min TV=0.000000 borne 1/(4q(T+1))=0.020833 vraie q>3(T+1)=False intermediaire=-0.187500


m=2 q= 5 E x1 A x2 (x1 <-> x2)    verite=0 min TV=0.008495 borne 1/(4q(T+1))=0.012500 ECHEC q>3(T+1)=False intermediaire=-0.058333


m=2 q= 5 A x1 E x2 (x1 <-> x2)    verite=1 min TV=0.000000 borne 1/(4q(T+1))=0.012500 vraie q>3(T+1)=False intermediaire=-0.058333
loi d'entree qui rend l'image exactement uniforme (instance fausse, q = 2) :
    (0, (0,), (0, 0, 0)) p = 0.250000
    (0, (0,), (1, 0, 0)) p = 0.250000
    (0, (1,), (0, 0, 0)) p = 0.250000
    (0, (1,), (1, 0, 0)) p = 0.250000
image M_S p : [0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125]  uniforme = 0.125


### Lecture du résultat — la promesse tient ou tombe avec le régime de $q$

Le programme linéaire cherche, sur **toutes** les lois d'entrée, la distance minimale à l'uniforme : $2\,TV(M_S p, U) = \sum_{u'} \bigl|(M_S p)(u') - 1/|\text{sorties}|\bigr|$, minimisée sur le simplexe. La dernière ligne du lemme 3.8 promet $TV \ge 1/(32(T+1)^2) = 1/(4q(T+1))$ — les deux écritures coïncident à $q = 8(T+1)$.

**Formules vraies** : la distance minimale est nulle — la loi honnête donne déjà l'image exactement uniforme. C'est le cas « oui ».

**Formules fausses**, trois régimes.

1. **$q = 2$ : la promesse est vide.** Sur la formule à une variable, la distance minimale vaut 0 : il existe une loi d'entrée qui rend l'image exactement uniforme bien que la formule soit fausse (`ECHEC`). La cellule l'affiche — quatre entrées de poids 0,25 chacune, $(0, \alpha, g)$ avec $\alpha \in \{0, 1\}$ et $g \in \{0, 1\}$, et une image à 0,125 sur chacune des huit sorties. Le menteur mélange à parts égales « tout zéro » et « tout un » ; à $q = 2$, l'échantillonneur ne le distingue plus d'une formule vraie. La formule à deux variables échoue de même.
2. **$q$ intermédiaire** (ici $q = 3$ et $q = 5$ pour $T = 1$) : la borne est respectée — 0,047 619 ≥ 0,041 667, puis 0,047 000 ≥ 0,025 000 — alors que la colonne `q>3(T+1)` dit `False`. La preuve du lemme 3.8 passe par une quantité intermédiaire, $\frac{1}{q+1}\left(\frac{1}{T+1} - \frac{3}{q}\right)$, encore **négative** (−0,125, puis −0,016 667) : la conclusion vaut sur ce jouet, mais l'argument de l'article ne couvre pas ce régime. Le carnet constate, il ne généralise pas.
3. **$q > 3(T+1)$** : l'intermédiaire devient positif (+0,008 929 à $q = 7$, +0,018 939 à $q = 11$) et la borne tient avec de la marge — 0,043 064 ≥ 0,017 857, puis 0,033 538 ≥ 0,011 364.

Les instances à deux variables ($T = 3$) exigeraient $q > 12$ : aux $q = 2$, $3$ et $5$ testés, elles échouent toutes les trois (0 ; 0,003 086 < 0,020 833 ; 0,008 495 < 0,012 500), comme le prédit le régime.

**Ce que cela enseigne.** La constante $q = 8(T+1)$ de l'article n'est pas un détail de rédaction : elle place d'emblée l'échantillonneur dans le troisième régime. Un échantillonneur construit sur une formule fausse n'est une instance « non » de `CIUniform` que si le corps est assez grand devant le nombre de tours.

## 3. Deux messages : le vérificateur stoquastique du Protocole 2

Le sum-check est devenu un échantillonneur. Il reste à montrer que `CIUniform` est dans la classe `qq-StoqAM` : qu'un vérificateur à **deux messages** — un message du vérificateur, un du prouveur — peut tester la promesse **avec des portes stoquastiques**.

### Ce que le mot contraint ici, et ce qu'il ne contraint pas

La définition de l'article porte sur le **circuit du vérificateur**, pas sur l'état du prouveur.

> **Definition 2.2 (Stoquastic verification circuit, implicit in [BBT06, Definition 4])** — « We say that a quantum circuit Q is a stoquastic verification circuit if Q acts on the registers (W, A0, A+), uses only Toffoli, CNOT, and X gates before the final measurement, and then measures the designated output qubit in the Hadamard basis. Here, the workspace register W contains an mw-qubit state, and the ancillary registers A0 and A+ are initialized to |0⟩⊗m0 and |+⟩⊗m+, respectively. The circuit Q accepts if and only if the final measurement gives outcome +. »

> **Definition 4.1 (qq-StoqAM)** — « … there exists an efficient quantum verifier V(x) = (V1(x), V2(x)) and a polynomial p, where V1 prepares p(n) EPR pairs and sends their second halves to the prover, and stores their first halves in V, and V2 is a stoquastic verification circuit as in Definition 2.2 acting on the message register M and V, such that:
> • Completeness. For every x ∈ Iyes, there exists a prover P(x) such that, upon receiving the halves of the EPR pairs, P(x) returns an ℓ(n)-qubit state and Pr[(P ⇌ V)(x) accepts] ≥ c(n).
> • Soundness. For any x ∈ Ino and any prover P(x) that returns an ℓ(n)-qubit state, Pr[(P ⇌ V)(x) accepts] ≤ s(n). »

Le prouveur rend **un état quantique quelconque** de $\ell(n)$ qubits : rien ne le limite. Ce qui est restreint, c'est ce que le vérificateur a le droit de faire de cet état — Toffoli, CNOT, X, puis une mesure de Hadamard. Et les paires d'EPR sont le « quantum public coin » de l'abstract : le vérificateur les prépare, garde une moitié, envoie l'autre.

### Le test qui fait tout le travail

L'article le reprend de Kim, Mančinska et Yuen — un test de SWAP entre l'image calculée et une référence fraîche, réalisé avec des portes qui n'ont que des $0$ et des $1$ hors diagonale.

> **Lemma 2.3 (SWAP test for mixed states, adapted from [KMY09, Proposition 9])** — « Let ρ0 and ρ1 be two m-qubit quantum states, which may in general be mixed. Then there exists a (2m + 1)-qubit stoquastic circuit that accepts with probability (1 + Tr(ρ0ρ1))/2, and uses a single copy of each ρ0 and ρ1, together with m Toffoli gates and 2m CNOT gates. »

### Le protocole

> **Protocol 2** (« A two-message stoquastic verifier for CIUniform[η] ») — « 1. Prepare mout EPR pairs across the registers (B, B″) and send B″ to the prover. 2. Receive the min-qubit input register A and the m+-qubit seed register R. Apply US† to the registers (A, R, B). 3. Apply the SWAP test between (R, B) and fresh reference registers (R′, B′) initialized to |+⟩⊗m+|0⟩⊗mout, using a control qubit O initialized to |+⟩. Accept if and only if measuring O in the Hadamard basis gives outcome +. »

$U_S$ est la version réversible de l'échantillonneur : $U_S\lvert u, r, w\rangle = \lvert u, r, w \oplus S(u, r)\rangle$, écrite par l'article avec « polynomially many Toffoli, CNOT, and X gates ». $U_S^\dagger$ annule donc exactement $B$ sur les termes où le prouveur a rendu $S(u, r)$ : après elle, la seule façon pour $B$ d'être dans $\lvert 0\ldots0\rangle$ est que le prouveur ait joué l'échantillonneur.

### Ce que le protocole accepte, et à quelle probabilité

> **Theorem 4.4** — « CIUniform[η] ∈ qq-StoqAM[1, 1 − η²/2]. »

Pour l'état $\rho$ sur $(A, R, B)$ reçu du prouveur, avec $\lvert\omega\rangle = \lvert+\rangle^{\otimes m_+}\lvert 0\rangle^{\otimes m_{out}}$ et $\Pi = U_S\,(I_A \otimes \lvert\omega\rangle\langle\omega\rvert)\,U_S^\dagger$ :

$$p_{acc} = \tfrac12\bigl(1 + \langle\omega\rvert\sigma\lvert\omega\rangle\bigr) = \tfrac12\bigl(1 + \mathrm{Tr}(\Pi\rho)\bigr), \qquad \sigma = \mathrm{Tr}_A\bigl(U_S^\dagger \rho\, U_S\bigr). \tag{4.1}$$

**Complétude** : si $M_S p = U$ pour une loi $p$, le prouveur renvoie $k_{u'} = 2^{(m_{out}-m_+)/2}\sum_{(u,r)\,:\,S(u,r)=u'}\sqrt{p(u)}\,\lvert u\rangle_A\lvert r\rangle_R$ ; $U_S^\dagger$ ramène tout sur $\lvert u\rangle_A\lvert\omega\rangle_{RB}$ et le test accepte avec certitude. **Solidité** : pour tout prouveur, la loi $p(u) = \lVert\chi_u\rVert^2$ qu'il définit vérifie $TV(M_S p, U) \le \sqrt{1 - \lvert\langle\psi\vert\xi\rangle\rvert^2}$, d'où $p_{acc} \le 1 - \eta^2/2$.

### Ce que ce carnet instancie, et les limites écrites

L'échantillonneur est simulé à $q = 2$ — le seul corps premier où les registres tiennent dans un ordinateur classique : $m_{in} = 5$, $m_+ = 2$, $m_{out} = 3$, soit 19 qubits. $U_S^\dagger$ y est appliquée **comme la permutation qu'elle est**, sans être décomposée en Toffoli et CNOT ; les portes du circuit, elles, sont simulées une par une. L'article, lui, prend $q = 8(T+1)$ — une puissance de deux, pas un corps premier — et compte ses portes « polynomially many ».

In [12]:
# --- 3a. Les portes du verificateur, et leur matrice explicite ---
H = NP.array([[1, 1], [1, -1]], dtype=complex) / NP.sqrt(2)
X = NP.array([[0, 1], [1, 0]], dtype=complex)


@functools.lru_cache(maxsize=None)
def bits_base(nb_qubits: int) -> NP.ndarray:
    """Matrice (2^nq, nq) de booleens : bits[k, q] = qubit q de l'etat de base k."""
    return ((NP.arange(2 ** nb_qubits)[:, None] >> NP.arange(nb_qubits)) & 1).astype(bool)


def porte(etat: NP.ndarray, matrice: NP.ndarray, cible: int | tuple[int, ...],
          controles: tuple[int, ...] = ()) -> None:
    """Applique en place une porte k-qubits (matrice 2^k x 2^k) aux qubits `cible`, commandee par `controles`.

    Les etats de base sont regroupes par valeurs des qubits non touches ; chaque groupe de 2^k
    amplitudes est multiplie par la matrice.
    """
    nb_qubits = etat.shape[0].bit_length() - 1
    bits = bits_base(nb_qubits)
    cible = tuple(cible)
    indices = NP.nonzero(NP.all(bits[:, list(controles)], axis=1))[0]
    autres = [k for k in range(nb_qubits) if k not in cible and k not in controles]
    cle = NP.zeros(len(indices), dtype=NP.int64)
    for k in autres:
        cle = (cle << 1) | bits[indices, k]
    pos = NP.zeros(len(indices), dtype=NP.int64)
    for k in cible:
        pos = (pos << 1) | bits[indices, k]
    ordre = NP.lexsort((pos, cle))
    blocs = etat[indices[ordre]].reshape(-1, matrice.shape[0])
    etat[indices[ordre]] = (blocs @ matrice.T).ravel()


def matrice_de_porte(nb_qubits: int, matrice: NP.ndarray, cible: int | tuple[int, ...],
                     controles: tuple[int, ...] = ()) -> NP.ndarray:
    """Matrice 2^nq x 2^nq de la porte telle que `porte` l'applique (colonnes = images de la base)."""
    M = NP.zeros((2 ** nb_qubits, 2 ** nb_qubits), dtype=complex)
    for j in range(2 ** nb_qubits):
        etat = NP.zeros(2 ** nb_qubits, dtype=complex)
        etat[j] = 1.0
        porte(etat, matrice, cible, controles)
        M[:, j] = etat
    return M


CNOT = NP.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 1], [0, 0, 1, 0]], dtype=complex)  # commande 0, cible 1
TOFFOLI = NP.zeros((8, 8), dtype=complex)                                                # commandes 0, 1 ; cible 2
for k in range(8):
    TOFFOLI[k ^ (((k & 1) & ((k >> 1) & 1)) << 2), k] = 1.0
CNOT_02 = NP.zeros((8, 8), dtype=complex)                                                # commande 0, cible 2
for k in range(8):
    CNOT_02[k ^ ((k & 1) << 2), k] = 1.0

print("=== 3a. le simulateur contre les matrices explicites (3 qubits) ===")
print("porte H sur le qubit 1        : ecart max =",
      float(NP.abs(matrice_de_porte(3, H, (1,)) - NP.kron(NP.kron(NP.eye(2), H), NP.eye(2))).max()))
print("porte CNOT (0 -> 2)           : ecart max =",
      float(NP.abs(matrice_de_porte(3, X, (2,), controles=(0,)) - CNOT_02).max()))
print("porte Toffoli (0, 1 -> 2)     : ecart max =",
      float(NP.abs(matrice_de_porte(3, X, (2,), controles=(0, 1)) - TOFFOLI).max()))
print("inventaire des entrees des portes du verificateur : H ->", sorted(set(NP.round(H.real.ravel(), 6))),
      "; X, CNOT, Toffoli ->",
      sorted(set(NP.round(NP.concatenate([X.real.ravel(), CNOT.real.ravel(), TOFFOLI.real.ravel()]), 6))))

=== 3a. le simulateur contre les matrices explicites (3 qubits) ===
porte H sur le qubit 1        : ecart max = 0.0
porte CNOT (0 -> 2)           : ecart max = 0.0
porte Toffoli (0, 1 -> 2)     : ecart max = 0.0
inventaire des entrees des portes du verificateur : H -> [np.float64(-0.707107), np.float64(0.707107)] ; X, CNOT, Toffoli -> [np.float64(0.0), np.float64(1.0)]


### Lecture du résultat (3a)

Les trois écarts sont **exactement nuls** — `0.0`, pas $10^{-16}$ : le simulateur ne fait que réindexer des amplitudes, il ne calcule aucune porte avec des nombres flottants. C'est ce qu'on veut avant de lui confier 19 qubits : le seul endroit où une erreur pourrait se glisser est le regroupement des états de base, et il est vérifié contre les matrices écrites à la main (Hadamard sur le qubit 1, CNOT 0→2, Toffoli 0,1→2).

L'inventaire des entrées dit ce que les portes sont :

- $H$ n'a que deux valeurs, $-0{,}707107$ et $+0{,}707107$ — c'est la **seule** porte du circuit qui sorte de $\{0, 1\}$ ;
- $X$, CNOT et Toffoli n'ont que $0$ et $1$ : ce sont des permutations de la base, donc des matrices à entrées réelles, calculables sans arithmétique flottante.

Cela ne suffit pas à dire « stoquastique » — le §4 montrera que le mot a **deux sens** (celui des matrices chez Bravyi-Bessen-Terhal, celui du circuit chez Liu) et que ce ne sont pas le même. Ce que la cellule établit, c'est le matériel du §3 : un registre de 19 qubits et des portes dont on connaît chaque entrée.

In [13]:
# --- 3b. Le circuit du Protocole 2 sur l'echantillonneur jouet, et les prouveurs ---
class Verificateur:
    """Le circuit a deux messages (Protocole 2) sur un echantillonneur jouet avec q = 2.

    Qubits, du bit de poids faible au plus fort : O ; B' ; R' ; B ; R ; A ; B''.
    Apres le message du prouveur, la suite du circuit ne touche plus B'' : le prouveur peut y laisser ce
    qu'il veut. U_S^dagger est appliquee comme la permutation qu'elle definit (non decomposee en portes).
    """

    def __init__(self, ech: Echantillonneur) -> None:
        self.ech = ech
        inst = ech.inst
        if inst.q != 2:
            raise ValueError("le circuit qubit exige q = 2")
        if (inst.T + 1) & inst.T:                       # T + 1 doit etre une puissance de deux
            raise ValueError("il faut T + 1 puissance de deux")
        ell = (inst.T + 1).bit_length() - 1             # log2(T + 1)
        self.m_in = ell + inst.m + 3                    # A : t, alpha, puis les trois coefficients
        self.m_plus = 2                                 # R et R' : la graine (r, v)
        self.m_out = ell + inst.m + 1                   # B, B', B'' : (t', alpha', nu)
        self.n_out = 2 ** self.m_out
        self.nb_graines = 2 ** self.m_plus
        debut_b = 1 + self.m_out + self.m_plus
        debut_a = debut_b + self.m_out + self.m_plus
        self.o = 0
        self.b_prime = tuple(range(1, 1 + self.m_out))
        self.r_prime = tuple(range(1 + self.m_out, debut_b))
        self.b = tuple(range(debut_b, debut_b + self.m_out))
        self.r = tuple(range(debut_b + self.m_out, debut_a))
        self.a = tuple(range(debut_a, debut_a + self.m_in))
        self.b_seconde = tuple(range(debut_a + self.m_in, debut_a + self.m_in + self.m_out))
        self.nb_qubits = debut_a + self.m_in + self.m_out
        self.table = ech.table().astype(NP.int64)       # S(u, graine) : indice de sortie
        indices = NP.arange(2 ** self.nb_qubits)
        cle_a = (indices >> self.a[0]) & (2 ** self.m_in - 1)            # contenu de A
        cle_r = (indices >> self.r[0]) & (2 ** self.m_plus - 1)          # contenu de R (graine)
        self.permutation = indices ^ (self.table[cle_a, cle_r] << self.b[0])   # U_S et U_S^dagger
        self.compteur: dict[str, int] = {}

    def _compte(self, nom: str) -> None:
        self.compteur[nom] = self.compteur.get(nom, 0) + 1

    def message(self, etat: NP.ndarray, isometrie: NP.ndarray,
                copie_privee: bool = False) -> NP.ndarray:
        """Le prouveur applique son isometrie V (lignes w = valeur de B'', colonnes (u, graine)) a B''."""
        blocs = etat.reshape(self.n_out, 2 ** self.m_in, self.nb_graines, -1)
        sortie = NP.zeros_like(blocs)
        recu = blocs[:, 0, 0, :]                          # avant le message : A = 0 et R = 0
        V = isometrie.reshape(self.n_out, 2 ** self.m_in, self.nb_graines)
        if copie_privee:
            for w in range(self.n_out):
                sortie[w] = NP.einsum("xg,z->xgz", V[w], recu[w])
        else:
            sortie[0] = NP.einsum("wxg,wz->xgz", V, recu)     # B'' est consomme : remis a |0>
        return sortie.reshape(-1)

    def deroule(self, isometrie: NP.ndarray,
                copie_privee: bool = False) -> tuple[NP.ndarray, NP.ndarray]:
        """Etat avant le test de SWAP, puis etat final juste avant la lecture de O."""
        self.compteur = {}
        etat = NP.zeros(2 ** self.nb_qubits, dtype=complex)
        etat[0] = 1.0
        for q, q2 in zip(self.b, self.b_seconde):       # 1. moitie d'EPR : CNOT sur |+>|0>
            porte(etat, H, (q,))
            porte(etat, X, (q2,), controles=(q,))
            self._compte("H de preparation")
            self._compte("CNOT")
        for q in self.r_prime:                          # registres de reference R' en |+>
            porte(etat, H, (q,))
            self._compte("H de preparation")
        etat = self.message(etat, isometrie, copie_privee)   # 2. message du prouveur
        etat = etat[self.permutation]                   # 2. U_S^dagger sur (A, R, B)
        avant_test = etat.copy()                        # O est encore dans |0> : la tranche O = 0 est exacte
        porte(etat, H, (self.o,))                       # 3. qubit de commande O en |+>
        self._compte("H de preparation")
        for source, cible in zip(self.r + self.b, self.r_prime + self.b_prime):   # 3. test de SWAP
            porte(etat, X, (cible,), controles=(source,))
            porte(etat, X, (source,), controles=(self.o, cible))
            porte(etat, X, (cible,), controles=(source,))
            self._compte("CNOT")
            self._compte("Toffoli")
            self._compte("CNOT")
        porte(etat, H, (self.o,))                       # 3. lecture de O dans la base de Hadamard
        self._compte("H de mesure")
        return avant_test, etat

    def recouvrement(self, avant_test: NP.ndarray) -> float:
        """<omega| rho_RB |omega> lu sur l'etat d'avant le test (equation (4.1))."""
        blocs = avant_test.reshape(self.n_out, 2 ** self.m_in, self.nb_graines, self.n_out,
                                   self.nb_graines, self.n_out, 2)   # axes B'', A, R, B, R', B', O
        amplitudes = blocs[:, :, :, 0, 0, 0, 0].sum(axis=2)           # B = 0, B' = 0, O = 0
        return float((NP.abs(amplitudes) ** 2).sum())

    def acceptation(self, isometrie: NP.ndarray, copie_privee: bool = False) -> float:
        """Probabilite que la mesure de O dans la base de Hadamard donne +."""
        _avant, etat = self.deroule(isometrie, copie_privee)
        return float((NP.abs(etat[0::2]) ** 2).sum())


def isometrie_de(ech: Echantillonneur, p: NP.ndarray) -> tuple[NP.ndarray, int]:
    """Isometrie du prouveur associee a une loi d'entree p (preuve du theoreme 4.4).

    Chaque sortie w recoit le vecteur |k_w> = somme_{(u,r) : S(u,r)=w} sqrt(p(u) / (graines * image(w)))
    |u>|r>, normalise par l'image de p. Une sortie que p ne produit jamais donne une ligne vide : on la
    complete par un vecteur de base d'une colonne inutilisee. Renvoie (V, nombre de lignes completees).
    """
    n_out = len(ech.sorties)
    table = ech.table()
    image = ech.matrice @ p
    V = NP.zeros((n_out, table.shape[0] * ech.nb_graines))
    for x in range(table.shape[0]):
        if p[x] == 0:
            continue
        for s in range(ech.nb_graines):
            V[table[x, s], x * ech.nb_graines + s] = NP.sqrt(p[x] / (ech.nb_graines * image[table[x, s]]))
    utilisees = NP.abs(V).sum(axis=0) > 0                 # colonnes deja prises
    completees = 0
    for w in range(n_out):
        if NP.linalg.norm(V[w]) < 1e-12:                 # ligne vide : sortie d'image nulle
            colonne = int(NP.flatnonzero(~utilisees)[0])
            V[w, colonne] = 1.0
            utilisees[colonne] = True
            completees += 1
    return V, completees


def isometrie_uniforme(ech: Echantillonneur) -> NP.ndarray:
    """Isometrie a coefficients constants sur les images reciproques des sorties."""
    table = ech.table()
    V = NP.zeros((len(ech.sorties), table.shape[0] * ech.nb_graines))
    for x in range(table.shape[0]):
        for s in range(ech.nb_graines):
            V[table[x, s], x * ech.nb_graines + s] = 1.0
    return V / NP.sqrt(V.sum(axis=1))[:, None]


def isometrie_aleatoire(ech: Echantillonneur, generateur: NP.random.Generator) -> NP.ndarray:
    """Isometrie aleatoire reelle (lignes orthonormales)."""
    nb_colonnes = len(ech.entrees) * ech.nb_graines
    bruit = generateur.standard_normal((nb_colonnes, len(ech.sorties)))
    Q, _R = NP.linalg.qr(bruit)
    return Q.T


def amplitudes_projetees(ech: Echantillonneur, isometrie: NP.ndarray) -> NP.ndarray:
    """Lambda_x = somme_s V[S(x, s), (x, s)] pour chaque entree x (une valeur par entree)."""
    table = ech.table()
    n_in = table.shape[0]
    blocs = isometrie.reshape(len(ech.sorties), n_in, ech.nb_graines)
    return blocs[table, NP.arange(n_in)[:, None], NP.arange(ech.nb_graines)[None, :]].sum(axis=1)


def loi_de_la_preuve(ech: Echantillonneur, isometrie: NP.ndarray) -> NP.ndarray:
    """Loi d'entree p(x) que la preuve du theoreme 4.4 attache a un prouveur : p(x) proportionnel a |Lambda_x|^2."""
    poids = NP.abs(amplitudes_projetees(ech, isometrie)) ** 2
    return poids / poids.sum()


def acceptation_reduite(ech: Echantillonneur, isometrie: NP.ndarray) -> tuple[float, float, float]:
    """Acceptation sans vecteur d'etat : (1 + F)/2 avec F = somme_x |Lambda_x|^2 / (q^2 n_out).

    Renvoie aussi le recouvrement F et la distance TV(M_S p, U) entre l'image de la loi de la preuve et
    la loi uniforme.
    """
    q, n_out = ech.inst.q, len(ech.sorties)
    poids = NP.abs(amplitudes_projetees(ech, isometrie)) ** 2
    recouvrement = float(poids.sum()) / (q * q * n_out)
    if poids.sum() <= 0:
        return 0.5, 0.0, float("nan")
    return 0.5 * (1.0 + recouvrement), recouvrement, ech.tv(loi_de_la_preuve(ech, isometrie))

### Le circuit, registre par registre

Le vérificateur tient dans 19 qubits, du bit de poids faible au plus fort :

- $O$ — qubit 0 : la commande du test de SWAP, préparée dans $\lvert+\rangle$ ;
- $B'$ — qubits 1 à 3 : la moitié « image » de la référence, initialisée à $\lvert 0\rangle^{\otimes 3}$ ;
- $R'$ — qubits 4 et 5 : la moitié « graine » de la référence, initialisée à $\lvert+\rangle^{\otimes 2}$ ;
- $B$ — qubits 6 à 8 : la moitié d'EPR gardée par le vérificateur, qui porte $u' = (t', \alpha', \nu)$ ;
- $R$ — qubits 9 et 10 : la graine reçue du prouveur ;
- $A$ — qubits 11 à 15 : l'entrée reçue du prouveur — $t$, $\alpha$, et les trois coefficients de $g$ ;
- $B''$ — qubits 16 à 18 : la moitié d'EPR envoyée au prouveur.

Trois choix d'implémentation, écrits pour qu'on ne les prenne pas pour l'article :

1. **$U_S^\dagger$ est appliquée comme la permutation qu'elle est**, $\lvert u, r, w\rangle \mapsto \lvert u, r, w \oplus S(u, r)\rangle$, et non décomposée en Toffoli et CNOT. L'article écrit que $U_S$ s'écrit avec « polynomially many Toffoli, CNOT, and X gates » ; le carnet applique le résultat, il ne refait pas la synthèse du circuit.
2. **Le prouveur consomme $B''$** : il remet sa moitié d'EPR à $\lvert 0\rangle$, comme le prouveur honnête de la preuve du théorème 4.4. Ce registre n'est plus touché ensuite, et c'est ce qui rend l'état d'avant le test lisible sur la tranche $O = 0$ — $O$ est encore dans $\lvert 0\rangle$ à cet instant.
3. **Les portes sont simulées une par une** : 6 Hadamard de préparation, 13 CNOT, 5 Toffoli, 1 Hadamard de mesure. Le relevé se lit en trois tranches : le test de SWAP du Lemme 2.3 pour $m = m_+ + m_{out} = 5$ — 5 Toffoli et 10 CNOT, plus deux Hadamard autour du qubit de commande ; la préparation des 3 paires d'EPR — 3 Hadamard, 3 CNOT ; la préparation des 2 ancillas $\lvert+\rangle$ de $R'$ — 2 Hadamard. Dans la définition 2.2, les ancillas $\lvert+\rangle$ sont des **entrées** et la lecture se fait dans la base de Hadamard : les Hadamard du relevé matérialisent des états d'entrée, ils ne sont pas des portes du circuit $V_2$. La cellule 3c imprime ce relevé et le confronte à la formule (4.1).

In [14]:
# 3c. Le circuit litteral, l'equation (4.1) et la formule reduite : trois voies vers le meme nombre.
# Instance vraie "E x1 (x1)" a q = 2 : 32 entrees, 4 graines, 8 sorties, donc 19 qubits.
inst_vrai = Instance(2, *FORMULES["E x1 (x1)"])
ech_vrai = Echantillonneur(inst_vrai)
verif = Verificateur(ech_vrai)
print("registres : m_in =", verif.m_in, "m_+ =", verif.m_plus, "m_out =", verif.m_out,
      "soit", verif.nb_qubits, "qubits ;", "graines =", verif.nb_graines, "sorties =", verif.n_out)
print("positions : O =", verif.o, "B' =", verif.b_prime, "R' =", verif.r_prime, "B =", verif.b,
      "R =", verif.r, "A =", verif.a, "B'' =", verif.b_seconde)

# Cinq prouveurs : l'isometrie honnete du Lemme 3.7, son image reciproque uniforme, trois temoins aleatoires.
# Les aleatoires tirent sur `generateur` (cree une seule fois dans la cellule d'imports) : l'ordre de tirage
# 3c -> 3f -> 4a bis -> 4b -> 4b bis est fige, c'est lui qui rejoue les memes chiffres d'une execution a l'autre.
strategies: dict[str, NP.ndarray] = {
    "honnete (p du Lemme 3.7)": isometrie_de(ech_vrai, ech_vrai.p_honnete())[0],
    "image reciproque uniforme": isometrie_uniforme(ech_vrai),
}
for k in range(3):
    strategies[f"aleatoire {k + 1}"] = isometrie_aleatoire(ech_vrai, generateur)

for nom, iso in strategies.items():
    # Voie 1 : le circuit, porte par porte, sur les 2^19 amplitudes. Voie 2 : l'equation (4.1), qui ne lit que
    # l'etat juste avant le test. Voie 3 : la formule reduite, qui ne connait que la loi p et l'image M_S p.
    reduite, recouvrement, distance = acceptation_reduite(ech_vrai, iso)
    avant, etat = verif.deroule(iso)
    litteral = float((NP.abs(etat[0::2]) ** 2).sum())
    formule = (1 + verif.recouvrement(avant)) / 2
    _avant_copie, etat_copie = verif.deroule(iso, copie_privee=True)
    litteral_copie = float((NP.abs(etat_copie[0::2]) ** 2).sum())
    # Une isometrie a ses lignes orthonormales : V V^T = I. C'est le temoin negatif du simulateur.
    ecart_orthogonalite = float(NP.abs(iso @ iso.T - NP.eye(verif.n_out)).max())
    print(f"{nom:28s} litteral={litteral:.12f} formule (4.1)={formule:.12f} "
          f"formule reduite={reduite:.12f} | F={recouvrement:.6f} TV(M_S p, U)={distance:.6f} "
          f"norme={float(NP.linalg.norm(etat)):.12f} lignes orthonormales : ecart={ecart_orthogonalite:.1e} "
          f"| prouveur qui garde une copie de u' : {litteral_copie:.12f}")

# L'ecart circuit / equation (4.1) isole, sur les cinq prouveurs : c'est la precision flottante, rien d'autre.
ecarts_trois_voies: list[float] = []
for iso in strategies.values():
    avant, etat = verif.deroule(iso)
    ecarts_trois_voies.append(
        abs(float((NP.abs(etat[0::2]) ** 2).sum()) - (1 + verif.recouvrement(avant)) / 2))
print(f"ecart max |litteral - formule (4.1)| sur les {len(strategies)} prouveurs "
      f"= {max(ecarts_trois_voies):.1e}")

# La loi p que le prouveur honnete fait apparaitre sur A : elle doit etre celle du Lemme 3.7.
loi_preuve = loi_de_la_preuve(ech_vrai, strategies["honnete (p du Lemme 3.7)"])
print("loi p de la preuve pour le prouveur honnete (poids > 0) :",
      [(ech_vrai.entrees[i], round(float(loi_preuve[i]), 6))
       for i in range(len(loi_preuve)) if loi_preuve[i] > 1e-12])
print(f"ecart max de la loi de la preuve avec p_honnete() = "
      f"{float(NP.abs(loi_preuve - ech_vrai.p_honnete()).max()):.1e}")

# Le releve des portes : il doit etre celui du Lemme 2.3 pour m = 5, plus les CNOT des paires d'EPR.
print("portes du verificateur :", verif.compteur)

registres : m_in = 5 m_+ = 2 m_out = 3 soit 19 qubits ; graines = 4 sorties = 8
positions : O = 0 B' = (1, 2, 3) R' = (4, 5) B = (6, 7, 8) R = (9, 10) A = (11, 12, 13, 14, 15) B'' = (16, 17, 18)


honnete (p du Lemme 3.7)     litteral=1.000000000000 formule (4.1)=1.000000000000 formule reduite=1.000000000000 | F=1.000000 TV(M_S p, U)=0.000000 norme=1.000000000000 lignes orthonormales : ecart=2.2e-16 | prouveur qui garde une copie de u' : 0.718750000000


image reciproque uniforme    litteral=0.990411805773 formule (4.1)=0.990411805773 formule reduite=0.990411805773 | F=0.980824 TV(M_S p, U)=0.072528 norme=1.000000000000 lignes orthonormales : ecart=3.3e-16 | prouveur qui garde une copie de u' : 0.708333333333


aleatoire 1                  litteral=0.516661413307 formule (4.1)=0.516661413307 formule reduite=0.516661413307 | F=0.033323 TV(M_S p, U)=0.234292 norme=1.000000000000 lignes orthonormales : ecart=6.7e-16 | prouveur qui garde une copie de u' : 0.516301506060


aleatoire 2                  litteral=0.516001263086 formule (4.1)=0.516001263086 formule reduite=0.516001263086 | F=0.032003 TV(M_S p, U)=0.143700 norme=1.000000000000 lignes orthonormales : ecart=7.8e-16 | prouveur qui garde une copie de u' : 0.514774066539


aleatoire 3                  litteral=0.511792975786 formule (4.1)=0.511792975786 formule reduite=0.511792975786 | F=0.023586 TV(M_S p, U)=0.105707 norme=1.000000000000 lignes orthonormales : ecart=6.7e-16 | prouveur qui garde une copie de u' : 0.511118293077


ecart max |litteral - formule (4.1)| sur les 5 prouveurs = 8.9e-16
loi p de la preuve pour le prouveur honnete (poids > 0) : [((0, (0,), (0, 1, 0)), 0.25), ((0, (1,), (0, 1, 0)), 0.25), ((1, (0,), (0, 0, 0)), 0.25), ((1, (1,), (0, 0, 0)), 0.25)]
ecart max de la loi de la preuve avec p_honnete() = 0.0e+00
portes du verificateur : {'H de preparation': 6, 'CNOT': 13, 'Toffoli': 5, 'H de mesure': 1}


### Lecture du résultat (3c) — trois voies, un seul nombre

La même probabilité d'acceptation sort de trois calculs indépendants. Le premier déroule les $2^{19}$ amplitudes porte par porte : les Hadamard, les CNOT, les Toffoli, puis la permutation $U_S^\dagger$ et la lecture de $O$ dans la base de Hadamard. Le second applique l'équation (4.1) à ce qui reste juste avant le test, $p_{acc} = \tfrac12\left(1 + \langle\omega\rvert\sigma\lvert\omega\rangle\right)$. Le troisième ne simule aucun état : il ne lit que la loi $p$ attachée au prouveur et son image $M_S p$, et calcule $p_{acc} = \tfrac12(1 + F)$ avec $F = \sum_x \lvert\Lambda_x\rvert^2 / (q^2 n_{out})$.

Les trois colonnes coïncident à douze décimales sur les cinq prouveurs ; l'écart maximal entre le circuit et l'équation (4.1) vaut $8{,}9 \times 10^{-16}$, c'est-à-dire la précision flottante. **Le simulateur de portes ne fait donc pas autre chose que l'algèbre de l'article** — et la formule réduite, qui ne connaît que les images, non plus.

**Le prouveur honnête accepte avec certitude.** $F = 1{,}000000$, $TV(M_S p, U) = 0{,}000000$, acceptation $1{,}000000000000$ : c'est la complétude parfaite du théorème 4.4, « $\mathrm{CIUniform}[\eta] \in qq\text{-}\mathrm{StoqAM}[1, 1 - \eta^2/2]$ », mesurée sur l'instance vraie. La loi que la preuve attache à ce prouveur est bien celle du Lemme 3.7 : les quatre entrées de poids $0{,}25$ qu'il fait apparaître sur $A$ sont exactement la loi honnête $p$ de la section 1, celle dont l'image est uniforme. La construction $\lvert\Lambda_x\rvert^2/(q^2 n_{out})$ la redonne terme à terme — l'écart avec `p_honnete()` est imprimé et vaut zéro à la précision flottante près.

**L'image réciproque uniforme passe sous la borne de solidité.** Son isométrie accepte $0{,}990412$ ($F = 0{,}980824$) ; or $1 - TV^2 = 0{,}994740$ et $1 - TV^2/2 = 0{,}997370$ pour $TV = 0{,}072528$. Les deux inégalités de la preuve du théorème 4.4 tiennent sur ce prouveur : $F \le 1 - TV^2$ d'abord, qui compare l'état préparé à la référence, puis $F \le 1 - TV^2/2$, la borne publiée. C'est par ce chemin que la solidité passe : un prouveur dont la loi s'écarte de l'uniforme d'une distance $TV$ ne peut pas dépasser $1 - TV^2/2$.

**Trois isométries réelles aléatoires tournent autour de $1/2$.** $0{,}516661$, $0{,}516001$, $0{,}511793$ : sans structure, un prouveur ne fait guère mieux que la pièce truquée. Leurs recouvrements sont minuscules ($F = 0{,}033323$, $0{,}032003$, $0{,}023586$) et leurs lignes restent orthonormales à $10^{-15}$ près — c'est le témoin négatif du simulateur, repris du §3a : une isométrie qui dériverait de $V V^\top = I$ ferait accepter n'importe quoi.

**Le relevé des portes se décompose en trois tranches.** $6$ Hadamard de préparation, $13$ CNOT, $5$ Toffoli, $1$ Hadamard de mesure. Les $5$ Toffoli et $10$ CNOT sont le test de SWAP du Lemme 2.3 pour $m = m_+ + m_{out} = 5$ — deux CNOT par qubit comparé, un Toffoli de commande — encadré de deux Hadamard sur $O$ ; les $3$ Hadamard et $3$ CNOT restants préparent les paires d'EPR ; les $2$ derniers Hadamard préparent les ancillas $\lvert+\rangle$ de $R'$. Dans la définition 2.2 ces ancillas sont des **entrées** et la lecture se fait en base de Hadamard : les Hadamard du relevé matérialisent des états, ils ne sont pas des portes de $V_2$. Ce que la définition contraint — Toffoli, CNOT, X — le relevé l'observe exactement.

**La dernière colonne demande une explication.** Un prouveur qui garde une copie de la valeur reçue n'accepte plus que $0{,}718750$ au lieu de $1$. C'est une observation du carnet, pas un énoncé de l'article : la cellule 3d en décompose le recouvrement valeur par valeur pour montrer d'où vient la perte.

**Ce que cette mesure ne dit pas.** La solidité n'est vérifiée ici que sur les cinq prouveurs testés ; le théorème, lui, quantifie sur **tous** les prouveurs. Ce que la cellule établit, c'est que la borne est **atteinte** par les constructions de la preuve et **non dépassée** par des témoins arbitraires — une vérification, pas une démonstration.

In [15]:
# 3d. La derniere colonne du 3c, expliquee : le test de SWAP ne lit que l'etat reduit de (R, B).
# On decompose son recouvrement avec |omega> = |++>_R |0..0>_B selon la valeur que B'' presente encore,
# puis on refait l'experience sans copie en projetant B sur une valeur choisie juste apres le message.
def recouvrement_par_valeur(verif: Verificateur,
                            avant_test: NP.ndarray) -> tuple[NP.ndarray, NP.ndarray]:
    """Recouvrement <omega| rho_RB^(w) |omega> de chaque secteur w de B'', et poids de ce secteur.

    Axes de `avant_test`, du bit de poids fort au plus faible : B'', A, R, B, R', B', O. La trace sur A et
    sur R' se prend sur psi psi* ; la projection sur |++>_R est la somme coherente sur les deux indices
    de R, divisee par le nombre de graines.
    """
    blocs = avant_test.reshape(verif.n_out, 2 ** verif.m_in, verif.nb_graines,
                               verif.n_out, verif.nb_graines, verif.n_out, 2)
    psi = blocs[:, :, :, 0, :, 0, 0]                       # B = 0, B' = 0, O = 0 ; reste (B'', A, R, R')
    noyau = NP.einsum("warh,waSh->wrS", psi, psi.conj())   # rho_R^(w)[r, S]
    return noyau.sum(axis=(1, 2)).real / verif.nb_graines, (NP.abs(psi) ** 2).sum(axis=(1, 2, 3))


class VerificateurAvecLecture(Verificateur):
    """Le meme circuit, avec une lecture fantome de B juste apres le message.

    La valeur de B est projetee sur un choix fixe : les autres sont annulees, sans renormaliser. Les poids,
    recouvrements et acceptations des secteurs s'additionnent alors, et leur somme doit redonner le resultat
    du circuit complet quand on parcourt les valeurs.
    """

    valeur_lue: int = 0

    def message(self, etat: NP.ndarray, isometrie: NP.ndarray,
                copie_privee: bool = False) -> NP.ndarray:
        sortie = super().message(etat, isometrie, copie_privee)   # le parent renvoie le nouvel etat
        indices = NP.arange(sortie.size)
        contenu = NP.zeros(sortie.size, dtype=NP.int64)
        for rang, qubit in enumerate(self.b):
            contenu |= ((indices >> qubit) & 1) << rang
        sortie[contenu != self.valeur_lue] = 0.0
        return sortie


iso_honnete = strategies["honnete (p du Lemme 3.7)"]
for etiquette, copie in (("honnete, sans copie", False), ("honnete, copie de u' gardee", True)):
    avant, _etat = verif.deroule(iso_honnete, copie_privee=copie)
    par_valeur, poids = recouvrement_par_valeur(verif, avant)
    print(f"{etiquette:28s} : poids par valeur de B'' = {NP.round(poids, 5).tolist()}")
    print(f"{'':28s}   recouvrement par valeur  = {NP.round(par_valeur, 5).tolist()} ; "
          f"somme = {float(par_valeur.sum()):.6f} ; "
          f"acceptation = {verif.acceptation(iso_honnete, copie):.12f}")

# Le controle : la meme chose sans copie, mais avec une lecture fantome de B sur chaque valeur possible.
lecteur = VerificateurAvecLecture(ech_vrai)
total_poids, total_recouvrement, total_acceptation = 0.0, 0.0, 0.0
for valeur in range(verif.n_out):
    lecteur.valeur_lue = valeur
    avant, etat = lecteur.deroule(iso_honnete)
    poids_secteur = float((NP.abs(avant) ** 2).sum())
    recouvrement_secteur = lecteur.recouvrement(avant)
    acceptation_secteur = float((NP.abs(etat[0::2]) ** 2).sum())
    total_poids += poids_secteur
    total_recouvrement += recouvrement_secteur
    total_acceptation += acceptation_secteur
    print(f"lecture de B = {valeur} : poids = {poids_secteur:.6f}, "
          f"recouvrement = {recouvrement_secteur:.6f}, acceptation = {acceptation_secteur:.6f}")
print(f"somme des {verif.n_out} lectures : poids = {total_poids:.6f}, "
      f"recouvrement = {total_recouvrement:.6f}, acceptation = {total_acceptation:.12f}")

honnete, sans copie          : poids par valeur de B'' = [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


                               recouvrement par valeur  = [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] ; somme = 1.000000 ; acceptation = 1.000000000000


honnete, copie de u' gardee  : poids par valeur de B'' = [0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125]


                               recouvrement par valeur  = [0.0625, 0.0625, 0.0625, 0.0625, 0.03125, 0.0625, 0.0625, 0.03125] ; somme = 0.437500 ; acceptation = 0.718750000000


lecture de B = 0 : poids = 0.125000, recouvrement = 0.062500, acceptation = 0.093750


lecture de B = 1 : poids = 0.125000, recouvrement = 0.062500, acceptation = 0.093750


... (2 outputs left out)

lecture de B = 4 : poids = 0.125000, recouvrement = 0.031250, acceptation = 0.078125


lecture de B = 5 : poids = 0.125000, recouvrement = 0.062500, acceptation = 0.093750


lecture de B = 6 : poids = 0.125000, recouvrement = 0.062500, acceptation = 0.093750


lecture de B = 7 : poids = 0.125000, recouvrement = 0.031250, acceptation = 0.078125
somme des 8 lectures : poids = 1.000000, recouvrement = 0.437500, acceptation = 0.718750000000


### Lecture du résultat (3d) — garder une copie, c'est mesurer

Sans copie, tout le recouvrement tombe dans **une seule** valeur de $B''$, la valeur $0$ : le prouveur honnête a consommé sa moitié d'EPR en la remettant à $\lvert 0\ldots0\rangle$, comme la preuve du théorème 4.4 le prescrit. Le recouvrement vaut alors $1$ et l'acceptation $1$ — la liste des poids est `[1, 0, 0, 0, 0, 0, 0, 0]`.

Avec une copie — le prouveur range la valeur reçue dans un registre privé au lieu de rendre sa moitié d'EPR — les huit valeurs reçues coexistent, **chacune dans un secteur orthogonal** : chaque secteur pèse $0{,}125$ et la somme des poids vaut bien $1$. Mais les recouvrements ne s'additionnent plus aussi bien : six secteurs donnent $0{,}0625$, deux donnent $0{,}03125$, pour une somme de $0{,}437500$ au lieu de $1$. L'acceptation retombe à $(1 + 0{,}4375)/2 = 0{,}718750$.

**D'où vient la perte.** Dans un secteur — une valeur reçue $w$ fixée — l'état rendu par le prouveur honnête ne reçoit de poids que des couples $(u, r)$ tels que $S(u, r) = w$ : c'est une **partie** de la graine, pas son tout. Le registre $R$ n'est dans $\lvert+\rangle^{\otimes m_+}$ qu'après la **somme cohérente sur toutes les valeurs reçues**, chaque couple $(u, r)$ apparaissant dans exactement un secteur. En gardant une copie, le prouveur sépare les secteurs : la somme cohérente devient une somme de probabilités, et les termes croisés qui restituaient $\lvert+\rangle_R$ disparaissent.

**L'expérience de contrôle le confirme.** La même décomposition, refaite sans copie mais en **projetant $B$ sur une valeur choisie** juste après le message, redonne exactement les mêmes nombres, secteur par secteur, et la somme des acceptations des huit lectures vaut $0{,}718750$. Autrement dit, `copie_privee=True` **équivaut** à une mesure de $B$ avant $U_S^\dagger$ — une lecture fantôme que le prouveur s'inflige à lui-même. C'est un prouveur **légal** au sens de la définition 4.1, rien ne lui interdit de garder une copie, et il perd la complétude parfaite.

**À quoi sert la référence quantique.** C'est le rôle de $\lvert\omega\rangle = \lvert+\rangle^{\otimes m_+}\lvert 0\rangle^{\otimes m_{out}}$ : la cohérence entre les valeurs reçues est ce que le test lit. Le prouveur honnête ne peut pas en garder trace sans la détruire, et c'est précisément pourquoi le protocole tient en deux messages.

Cette observation est celle du carnet ; l'article ne l'énonce pas sous cette forme.

In [16]:
# 3e. L'instance fausse a q = 2 : la promesse "CIUniform" y est vide, et cela se voit.
# Le meilleur prouveur possible est celui de la loi d'entree qui rend l'image exactement uniforme.
inst_faux = Instance(2, *FORMULES["A x1 (x1)"])
ech_faux = Echantillonneur(inst_faux)
verif_faux = Verificateur(ech_faux)
tv_min, p_etoile = ech_faux.tv_minimal()

# Deux prouveurs : celui de la meilleure loi (il realise l'uniforme, donc accepte a coup sur), et l'honnete.
# `lignes completees` compte les lignes vides de son isometrie qu'il a fallu completer pour la rendre physique.
candidats = (
    ("isometrie de la meilleure loi", isometrie_de(ech_faux, p_etoile)),
    ("prouveur honnete", isometrie_de(ech_faux, ech_faux.p_honnete())),
)
for nom, (iso, completees) in candidats:
    avant, etat = verif_faux.deroule(iso)
    litteral = float((NP.abs(etat[0::2]) ** 2).sum())
    formule = (1 + verif_faux.recouvrement(avant)) / 2
    reduite, recouvrement, distance = acceptation_reduite(ech_faux, iso)
    ecart_orthogonalite = float(NP.abs(iso @ iso.T - NP.eye(verif_faux.n_out)).max())
    print(f"min TV={tv_min:.6f} ; {nom:30s} : litteral={litteral:.12f} "
          f"formule (4.1)={formule:.12f} formule reduite={reduite:.12f} | "
          f"F={recouvrement:.6f} TV(M_S p, U)={distance:.6f} ; lignes completees={completees} ; "
          f"V V^T - I : ecart={ecart_orthogonalite:.1e}")

min TV=0.000000 ; isometrie de la meilleure loi  : litteral=1.000000000000 formule (4.1)=1.000000000000 formule reduite=1.000000000000 | F=1.000000 TV(M_S p, U)=0.000000 ; lignes completees=0 ; V V^T - I : ecart=2.2e-16


min TV=0.000000 ; prouveur honnete               : litteral=0.864276695297 formule (4.1)=0.864276695297 formule reduite=0.864276695297 | F=0.728553 TV(M_S p, U)=0.250000 ; lignes completees=2 ; V V^T - I : ecart=2.2e-16


### Lecture du résultat (3e) — une instance hors promesse

À $q = 2$, la distance minimale sur toutes les lois d'entrée vaut $0{,}000000$ : la promesse de `CIUniform` est **vide** sur cette instance fausse, comme le §2c l'avait mesuré. La cellule reprend ce résultat pour construire un prouveur.

**Un prouveur qui accepte à coup sûr, et ce n'est pas un défaut du simulateur.** L'isométrie de la meilleure loi rend l'image exactement uniforme : $F = 1{,}000000$, $TV = 0{,}000000$, acceptation $1{,}000000000000$ par les trois voies. Le théorème 4.4 ne dit rien de cette instance : il suppose $x \in I_{no}$, et une instance dont la distance minimale est nulle n'est pas dans $I_{no}$. C'est le jouet qui est hors promesse, pas le théorème qui est en défaut.

**Le prouveur honnête, lui, ne passe plus.** $0{,}864277$ d'acceptation, $F = 0{,}728553$, $TV = 0{,}250000$ — exactement la distance $1/(q(T+1))$ du Lemme 3.7 pour $q = 2$ et $T = 1$. Sur une formule fausse, le prouveur qui suit le protocole n'a aucune raison d'être accepté.

**Deux lignes d'isométrie à compléter.** Le compteur vaut $0$ pour la meilleure loi et $2$ pour le prouveur honnête : la loi honnête d'une instance fausse ne produit que six des huit sorties, deux lignes de $V$ restent vides, et une ligne vide n'est pas une isométrie. La cellule les complète par un vecteur de base d'une colonne libre. C'est un choix d'implémentation, écrit ici : la construction de la preuve suppose une image uniforme, pas la loi honnête d'une instance fausse. Après complétion les deux isométries restent orthonormales à $2{,}2 \times 10^{-16}$.

**Ce que la complétion change, et ce qu'elle ne change pas.** Entre le circuit littéral et la formule réduite, les deux prouveurs donnent le même nombre à douze décimales : la complétion n'introduit aucun écart caché entre les voies. Une isométrie **non** complétée, elle, n'est pas physique : l'exercice 2, juste après, fait mesurer l'écart que cela produit.

### Exercice 2 — Compléter les lignes vides, ou ne pas les compléter

`isometrie_de` remplit chaque ligne de $V$ par la formule de la preuve du théorème 4.4, puis complète les lignes restées vides par un vecteur de base d'une colonne libre. Sur l'instance fausse à $q = 2$, deux lignes sont vides : la loi honnête ne charge que six des huit sorties.

1. Reconstruisez $V$ par la même formule, **sans** compléter les lignes vides. Le squelette est donné : `V_exo` est déjà dimensionné.
2. Mesurez $\lVert V V^{\mathsf{T}} - I \rVert_{\max}$ pour votre matrice et pour celle que renvoie `isometrie_de`.
3. Passez l'une et l'autre dans `verif_faux.deroule`. Comparez la probabilité du circuit à la formule réduite de la cellule 3c.

**Ce qu'il faut regarder.** Où l'écart à l'identité est-il non nul, et combien de lignes sont en cause ? Les deux prouveurs donnent-ils la même probabilité ? Si non, lequel des deux reste égal à la formule réduite — et quelle est la norme de l'état en sortie du circuit dans chaque cas ?

In [17]:
# Exercice 2 : la meme isometrie que isometrie_de, sans completer les lignes vides
table_exo = ech_faux.table()
p_exo = ech_faux.p_honnete()
image_exo = ech_faux.matrice @ p_exo
V_exo = NP.zeros((len(ech_faux.sorties), table_exo.shape[0] * ech_faux.nb_graines))
# TODO etudiant
# Indice : pour chaque entree x de poids p_exo[x] > 0 et chaque graine, une seule case de la ligne
# table_exo[x, graine] est non nulle ; sa valeur est donnee par la formule de la cellule 3b.
# Etape 1 : remplir V_exo (pas de completion des lignes vides).
# Etape 2 : mesurer l'ecart a l'identite pour V_exo et pour isometrie_de(ech_faux, p_exo)[0].
# Etape 3 : passer les deux dans verif_faux.deroule et comparer a acceptation_reduite(ech_faux, V).
V_reference, completees_exo = isometrie_de(ech_faux, p_exo)
for nom, V in (("sans completion", V_exo), ("isometrie_de", V_reference)):
    _avant, etat_exo = verif_faux.deroule(V)
    print(f"{nom:16s} : |V V^T - I| max = {float(NP.abs(V @ V.T - NP.eye(V.shape[0])).max()):.3e} ; "
          f"norme en sortie du circuit = {float(NP.linalg.norm(etat_exo)):.6f} ; "
          f"acceptation du circuit = {float((NP.abs(etat_exo[0::2]) ** 2).sum()):.12f}")
print("Exercice 2 a completer : lignes vides de V_exo =",
      int((NP.abs(V_exo).sum(axis=1) == 0).sum()), "sur", V_exo.shape[0])

sans completion  : |V V^T - I| max = 1.000e+00 ; norme en sortie du circuit = 0.000000 ; acceptation du circuit = 0.000000000000


isometrie_de     : |V V^T - I| max = 2.220e-16 ; norme en sortie du circuit = 1.000000 ; acceptation du circuit = 0.864276695297
Exercice 2 a completer : lignes vides de V_exo = 8 sur 8


### Trois corps de plus, et le compte des copies

Le protocole à deux messages a jusqu'ici été rejoué dans un seul corps, $\mathbb{F}_2$, sur une formule à une variable, vraie puis fausse — et le circuit qubit par qubit ne l'a été que là. La cellule 3f reprend le balayage en corps du §2c, $q \in \{2, 3, 5, 7\}$, par la formule réduite de la cellule 3c : aucune ligne ne rejoue le circuit.

Elle ajoute une dernière colonne : le nombre de copies du lemme 4.2. $r$ répétitions en parallèle font passer un seuil $s$ à $(1 + s^r)/2$, qui passe sous $3/4$ dès que $s^r \le 1/2$. C'est le seul levier d'amplification que laisse un vérificateur stoquastique — la section 4 y revient.

In [18]:
# 3f. Balayage de q sur les deux instances a m = 1 : ou la borne 1 - TV^2 / 2 mord-elle vraiment ?
# "copies pour 3/4" repond a une question de repetition parallele (Lemme 4.2) : combien de copies du
# protocole faut-il enchainer pour que le meilleur prouveur retombe sous 3/4 ?
for nom in ("E x1 (x1)", "A x1 (x1)"):
    for q in (2, 3, 5, 7):
        inst = Instance(q, *FORMULES[nom])
        ech = Echantillonneur(inst)
        tv_min, p_etoile = ech.tv_minimal()
        nb_aleas = 3 if q == 7 else 10
        aleas = [acceptation_reduite(ech, isometrie_aleatoire(ech, generateur)) for _ in range(nb_aleas)]
        borne = 1 - tv_min ** 2 / 2
        iso_h, completees_h = isometrie_de(ech, ech.p_honnete())
        iso_e, completees_e = isometrie_de(ech, p_etoile)
        lignes = {
            "honnete": acceptation_reduite(ech, iso_h),
            "meilleure_loi": acceptation_reduite(ech, iso_e),
            "image_reciproque": acceptation_reduite(ech, isometrie_uniforme(ech)),
            f"max_{nb_aleas}_aleatoires": max(aleas),
        }
        pire = max(valeur[0] for valeur in lignes.values())
        # Lemme 4.2 : r copies independantes donnent une solidite (1 + pire^r) / 2. Elle passe sous 3/4 des que
        # pire^r <= 1/2, d'ou r = log(1/2) / log(pire), arrondi au-dessus. Infini quand pire = 1 : la completude
        # est parfaite sur une instance vraie, ou la promesse est vide sur la fausse instance a q = 2 ; dans les
        # deux cas aucune copie ne fait bouger le nombre.
        r_copie = (NP.log(0.5) / NP.log(pire)) if 0 < pire < 1 else float("inf")
        ortho = max(float(NP.abs(une_iso @ une_iso.T - NP.eye(len(ech.sorties))).max())
                    for une_iso in (iso_h, iso_e))
        print(f"{nom:12s} q={q} verite={valeur_de_verite(inst)} min TV={tv_min:.6f} "
              f"borne 1-TV^2/2={borne:.8f} "
              + " ".join(f"{cle}={valeur[0]:.8f}" for cle, valeur in lignes.items())
              + f" | max={pire:.8f} <= borne: {pire <= borne + 1e-12} | F <= 1-TV^2 partout: "
                f"{all(valeur[1] <= 1 - valeur[2] ** 2 + 1e-12 for valeur in lignes.values())} "
                f"| copies pour 3/4 : {int(NP.ceil(r_copie)) if NP.isfinite(r_copie) else 'infini'}"
                f" | lignes completees honnete={completees_h} meilleure_loi={completees_e} ; "
                f"V V^T - I : {ortho:.1e}")

E x1 (x1)    q=2 verite=1 min TV=0.000000 borne 1-TV^2/2=1.00000000 honnete=1.00000000 meilleure_loi=1.00000000 image_reciproque=0.99041181 max_10_aleatoires=0.52205671 | max=1.00000000 <= borne: True | F <= 1-TV^2 partout: True | copies pour 3/4 : infini | lignes completees honnete=0 meilleure_loi=0 ; V V^T - I : 2.2e-16
E x1 (x1)    q=3 verite=1 min TV=0.000000 borne 1-TV^2/2=1.00000000 honnete=1.00000000 meilleure_loi=1.00000000 image_reciproque=0.98552003 max_10_aleatoires=0.50374907 | max=1.00000000 <= borne: True | F <= 1-TV^2 partout: True | copies pour 3/4 : infini | lignes completees honnete=0 meilleure_loi=0 ; V V^T - I : 1.1e-16


E x1 (x1)    q=5 verite=1 min TV=0.000000 borne 1-TV^2/2=1.00000000 honnete=1.00000000 meilleure_loi=1.00000000 image_reciproque=0.98290854 max_10_aleatoires=0.50044014 | max=1.00000000 <= borne: True | F <= 1-TV^2 partout: True | copies pour 3/4 : infini | lignes completees honnete=0 meilleure_loi=0 ; V V^T - I : 4.4e-16


E x1 (x1)    q=7 verite=1 min TV=0.000000 borne 1-TV^2/2=1.00000000 honnete=1.00000000 meilleure_loi=1.00000000 image_reciproque=0.98380373 max_3_aleatoires=0.50010647 | max=1.00000000 <= borne: True | F <= 1-TV^2 partout: True | copies pour 3/4 : infini | lignes completees honnete=0 meilleure_loi=0 ; V V^T - I : 3.3e-16
A x1 (x1)    q=2 verite=0 min TV=0.000000 borne 1-TV^2/2=1.00000000 honnete=0.86427670 meilleure_loi=1.00000000 image_reciproque=0.97311359 max_10_aleatoires=0.51947565 | max=1.00000000 <= borne: True | F <= 1-TV^2 partout: True | copies pour 3/4 : infini | lignes completees honnete=2 meilleure_loi=0 ; V V^T - I : 2.2e-16
A x1 (x1)    q=3 verite=0 min TV=0.047619 borne 1-TV^2/2=0.99886621 honnete=0.90713484 meilleure_loi=0.99113987 image_reciproque=0.97464974 max_10_aleatoires=0.50401379 | max=0.99113987 <= borne: True | F <= 1-TV^2 partout: True | copies pour 3/4 : 78 | lignes completees honnete=3 meilleure_loi=0 ; V V^T - I : 2.2e-16


A x1 (x1)    q=5 verite=0 min TV=0.047000 borne 1-TV^2/2=0.99889550 honnete=0.94313708 meilleure_loi=0.99083034 image_reciproque=0.97916267 max_10_aleatoires=0.50043098 | max=0.99083034 <= borne: True | F <= 1-TV^2 partout: True | copies pour 3/4 : 76 | lignes completees honnete=5 meilleure_loi=0 ; V V^T - I : 4.4e-16


A x1 (x1)    q=7 verite=0 min TV=0.043064 borne 1-TV^2/2=0.99907273 honnete=0.95903348 meilleure_loi=0.99073603 image_reciproque=0.98200472 max_3_aleatoires=0.50010604 | max=0.99073603 <= borne: True | F <= 1-TV^2 partout: True | copies pour 3/4 : 75 | lignes completees honnete=7 meilleure_loi=0 ; V V^T - I : 6.7e-16


### Lecture du résultat (3f) — où la borne mord, et ce qu'il faut pour amplifier

Deux instances à $m = 1$, quatre corps de base, quatre familles de prouveurs par ligne : honnête, meilleure loi, image réciproque uniforme, et le maximum de dix isométries aléatoires (trois à $q = 7$).

**Instances vraies.** La distance minimale est nulle à tous les $q$ ; l'honnête et la meilleure loi acceptent tous deux avec $1{,}00000000$ — quand la promesse est satisfaite, la meilleure loi *est* la loi honnête. L'image réciproque uniforme reste haute sans atteindre $1$ : $0{,}99041$, $0{,}98552$, $0{,}98291$, $0{,}98380$. Les aléatoires plafonnent à $0{,}5221$, $0{,}5037$, $0{,}5004$, $0{,}5001$ : la structure de l'instance ne donne presque rien à un prouveur qui ne la tient pas.

**Instances fausses : $q = 2$ est hors promesse.** La distance minimale vaut $0$, la meilleure loi accepte avec $1$ — la borne $1 - TV^2/2$ vaut alors $1$ et ne contraint rien.

**Instances fausses : $q \ge 3$.** La distance minimale vaut $0{,}047619$ ($q = 3$), $0{,}047000$ ($q = 5$), $0{,}043064$ ($q = 7$) — à chaque fois au-dessus de $1/(4q(T+1))$, comme le Lemme 3.8 le promet. La borne $1 - TV^2/2$ vaut $0{,}99886621$, $0{,}99889550$, $0{,}99907273$ : presque $1$, donc presque rien. Le meilleur prouveur testé atteint $0{,}99113987$, $0{,}99083034$, $0{,}99073603$ — **sous la borne**, mais loin au-dessus de $3/4$. La marge de solidité sur ce jouet est inférieure à $1~\%$ : c'est elle que la répétition parallèle doit creuser.

**Combien de copies ?** Sous le Lemme 4.2, $r$ copies indépendantes donnent une solidité $(1 + p^r)/2$ où $p$ est la probabilité d'acceptation d'une copie. Elle passe sous $3/4$ dès que $p^r \le 1/2$, soit $r = \lceil \log(1/2)/\log p \rceil$ : $78$ copies à $q = 3$, $76$ à $q = 5$, $75$ à $q = 7$. Sur les instances vraies, comme sur la fausse à $q = 2$, le nombre affiché est « infini » — et c'est le même signe dans les deux cas : $1^r = 1$, aucune répétition ne fait descendre une acceptation de $1$. Sur une instance vraie c'est la complétude parfaite, qu'on veut garder ; sur la fausse à $q = 2$ c'est la promesse vide, qui ne dit rien.

**Les invariants tiennent sur les huit lignes.** `F <= 1-TV^2 partout` vaut `True` partout, l'orthogonalité des isométries tient à $10^{-16}$ près, et `lignes completees` vaut $2$, $3$, $5$, $7$ pour l'honnête sur l'instance fausse, $0$ pour la meilleure loi dont l'image est uniforme : la loi honnête d'une instance fausse laisse des sorties vides, et leur nombre croît avec $q$.

**La solidité reste vérifiée, pas démontrée.** Les prouveurs testés forment quatre familles ; le théorème quantifie sur tous. Ce que la cellule établit, c'est que la borne de l'article n'est franchie ni par les constructions de sa preuve ni par des témoins arbitraires, et que la marge à combler par répétition est de l'ordre du pour cent sur ce jouet.

## 4. « Stoquastique » : deux sens, et ce que la contrainte coûte

Le mot vient de Bravyi, Bessen et Terhal (2006), et l'article en fait **deux** usages. La section 3 a cité la définition 2.2 de Liu, qui est la version interactive du second. Il faut les séparer, parce qu'ils ne contraignent pas la même chose.

### Premier sens : une matrice, et un état fondamental sans signe

> **Definition 2 (Bravyi–Bessen–Terhal 2006)** — « Stoquastic LH-MIN is defined as LH-MIN with the restriction that all operators Hα have real non-positive off-diagonal matrix elements in the computational basis. »

L'article dit aussitôt ce que la restriction achète :

> « The important consequence of this restriction is that the eigenvector with lowest eigenvalue, also called the ground-state, of a Hamiltonian H = Σα Hα is a vector with nonnegative coefficients in the computational basis. This allows for an interpretation of this vector as a probability distribution. For a general Hamiltonian the ground-state is a vector with complex coefficients for which no such representation exists. »

Un état fondamental à coefficients tous positifs ou nuls se lit comme une distribution de probabilité. La cellule 4a vérifie la conséquence sur des matrices de trois sites ; la cellule 4a bis la teste sur deux mille matrices de quatre sites et mesure, dans le même mouvement, ce qu'elle coûte.

### Second sens : un vérificateur, et des portes sans signe

> **Definition 4 (Bravyi–Bessen–Terhal 2006)** — « A stoquastic verifier is a tuple V = (n, nw, n0, n+, U), where n is the number of input bits, nw the number of input witness qubits, n0 the number of input ancillas |0⟩, n+ the number of input ancillas |+⟩ and U is a quantum circuit on n + nw + n0 + n+ qubits with X, CNOT, and Toffoli gates. »

L'article résume la classe d'une phrase :

> « The class StoqMA is a restricted version of QMA in which the verifier can perform only classical reversible gates, prepare qubits in |0 and |+ states, and perform one measurement in the |+ , |− basis. »

Les trois portes autorisées sont des permutations : dans la base de calcul, leurs matrices n'ont hors diagonale que des $0$ et des $1$, jamais de signe moins. Le vérificateur ne fait donc rien d'autre qu'une permutation entre l'entrée et la mesure ; sa seule liberté par rapport à un vérificateur classique cohérent est la **base dans laquelle il lit** :

> « It is important to note that the only difference between StoqMA and MA is that a stoquastic verifier in StoqMA is allowed to do the final measurement in the {|+⟩, |−⟩} basis, whereas a classical coherent verifier in MA can only do a measurement in the standard basis {|0⟩, |1⟩}. »

### Ce que le mot ne dit pas, et ce qu'il coûte

- **Il ne contraint pas le prouveur.** Le témoin de la définition 4 est un état quelconque ; dans la définition 4.1 de Liu, le prouveur est un circuit quantique arbitraire. La contrainte pèse entièrement sur le vérificateur.
- **Il n'interdit pas la superposition : il la renvoie aux bords.** Les ancillas entrent dans l'état $\lvert+\rangle$ et la lecture se fait dans la base de Hadamard. Entre les deux, le circuit permute.
- **Il coûte l'amplification par vote.** L'article de 2006 est explicite : « In contrast to the standard classes BPP, MA, or QMA the class StoqMA does not permit amplification of the gap between the threshold probabilities ǫno, ǫyes based on majority voting. » Le lemme 4.2 de Liu en fournit une autre, pour les protocoles à deux messages : $r$ répétitions en parallèle ramènent le seuil de solidité à $(1 + s^r)/2$. C'est cette formule dont la cellule 3f a tiré son compte de copies.
- **Il coûte, côté preuve, l'hypothèse d'un témoin positif.** Sans interaction, l'article de 2006 remarque que « the state ψ maximizing the acceptance probability has non-negative amplitudes in the computational basis » : le témoin optimal peut être pris positif. Avec interaction, la remarque 4.3 de Liu montre que cette hypothèse devient fausse : un état joint à amplitudes négatives fait accepter un vérificateur stoquastique avec une probabilité strictement plus grande. Le vérificateur subit des signes qu'il ne peut pas produire lui-même.

Les cellules qui suivent mesurent ces deux coûts : 4a et 4a bis pour le premier sens, l'énergie d'une matrice dont les signes ne se laissent pas ramener tous au même signe ; 4b et 4b bis pour le second, la probabilité d'acceptation d'un vérificateur stoquastique face à des prouveurs à états positifs ou non ; 4c pour les signes que porte effectivement le circuit de la section 3.

In [19]:
# 4a. Le premier sens sur des matrices de trois sites : ce que la definition garantit, et ce qu'elle ne dit pas

def est_stoquastique(Hm: NP.ndarray, tolerance: float = 1e-12) -> bool:
    """Definition 2 de Bravyi, Bessen et Terhal : termes reels, tous <= 0 hors de la diagonale."""
    hors_diagonale = Hm - NP.diag(NP.diag(Hm))
    return bool(NP.all(NP.abs(NP.imag(hors_diagonale)) <= tolerance)
                and NP.all(NP.real(hors_diagonale) <= tolerance))


def jauge_stoquastique(Hm: NP.ndarray, tolerance: float = 1e-12) -> NP.ndarray | None:
    """Signes (s_i) tels que s_i s_j H_ij <= 0 hors diagonale, ou None.

    Multiplier la ligne i et la colonne i par s_i remplace chaque vecteur de base par s_i fois lui-meme :
    les valeurs propres ne changent pas, les composantes de l'etat fondamental sont multipliees par s_i,
    et la propriete d'etre sans signe est conservee.
    """
    n = Hm.shape[0]
    for motif in itertools.product((1.0, -1.0), repeat=n):
        signes = NP.array(motif)
        if est_stoquastique(signes[:, None] * signes[None, :] * Hm.real, tolerance):
            return signes
    return None


def cousin_stoquastique(Hm: NP.ndarray) -> NP.ndarray:
    """Meme diagonale, memes magnitudes hors diagonale, tous les termes hors diagonale negatifs."""
    diagonale = NP.diag(NP.diag(Hm))
    return diagonale - (NP.abs(Hm) - NP.diag(NP.diag(NP.abs(Hm))))


def etat_fondamental(Hm: NP.ndarray) -> tuple[NP.ndarray, float]:
    """Vecteur propre de plus petite valeur propre, signe fixe par sa plus grande composante."""
    valeurs, vecteurs = NP.linalg.eigh(Hm)
    g = vecteurs[:, 0]
    return (g if g[NP.argmax(NP.abs(g))] > 0 else -g), float(valeurs[1] - valeurs[0])


def signe_libre(v: NP.ndarray, tolerance: float = 1e-9) -> bool:
    """Vrai si toutes les entrees de v sont >= 0, ou toutes <= 0 : l'etat n'a qu'un signe."""
    return bool(v.min() >= -tolerance or v.max() <= tolerance)


CHAINE = NP.array([[0.0, -1.0, 0.0], [-1.0, 0.0, -1.0], [0.0, -1.0, 0.0]])
TRIANGLE = NP.array([[0.0, -1.0, -1.0], [-1.0, 0.0, -1.0], [-1.0, -1.0, 0.0]])
BOUCLE = NP.array([[0.0, -1.0, 1.0], [-1.0, 0.0, -1.0], [1.0, -1.0, -0.4]])   # triangle, une liaison positive
COUSIN = cousin_stoquastique(BOUCLE)            # memes magnitudes, tous les signes hors diagonale rendus negatifs
for nom, Hm in (("chaine -1", CHAINE), ("triangle -1", TRIANGLE), ("boucle (-, -, +)", BOUCLE),
                ("cousin stoquastique", COUSIN)):
    g, ecart_spectral = etat_fondamental(Hm)
    produit_boucle = Hm[0, 1] * Hm[1, 2] * Hm[2, 0]        # invariant par jauge : les s_i entrent au carre
    ecart_cousin = float(NP.abs(cousin_stoquastique(Hm) - Hm).max())
    print(f"{nom:20s} stoquastique={str(est_stoquastique(Hm)):5s} "
          f"a jauge pres={str(jauge_stoquastique(Hm) is not None):5s} produit de boucle={produit_boucle:+.1f} "
          f"valeurs propres={NP.round(NP.linalg.eigvalsh(Hm), 4)}")
    print(f"{'':20s} etat fondamental={NP.round(g, 4)} sans signe={signe_libre(g)} "
          f"ecart spectral={ecart_spectral:.4f} cousin - H = {ecart_cousin:.1e}")
for eps in (0.4, -0.4, 1.0, -1.0):
    Hm = NP.array([[0.0, -1.0, 1.0], [-1.0, 0.0, -1.0], [1.0, -1.0, eps]])
    g, ecart_spectral = etat_fondamental(Hm)
    print(f"site 3 a {eps:+.1f} : a jauge pres={jauge_stoquastique(Hm) is not None} "
          f"valeurs propres={NP.round(NP.linalg.eigvalsh(Hm), 4)} etat fondamental={NP.round(g, 4)} "
          f"sans signe={signe_libre(g)} ecart spectral={ecart_spectral:.4f}")

chaine -1            stoquastique=True  a jauge pres=True  produit de boucle=+0.0 valeurs propres=[-1.4142 -0.      1.4142]
                     etat fondamental=[0.5    0.7071 0.5   ] sans signe=True ecart spectral=1.4142 cousin - H = 0.0e+00
triangle -1          stoquastique=True  a jauge pres=True  produit de boucle=-1.0 valeurs propres=[-2.  1.  1.]
                     etat fondamental=[0.5774 0.5774 0.5774] sans signe=True ecart spectral=3.0000 cousin - H = 0.0e+00
boucle (-, -, +)     stoquastique=False a jauge pres=False produit de boucle=+1.0 valeurs propres=[-1.278 -1.     1.878]
                     etat fondamental=[-0.373   0.373   0.8496] sans signe=False ecart spectral=0.2780 cousin - H = 2.0e+00
cousin stoquastique  stoquastique=True  a jauge pres=True  produit de boucle=-1.0 valeurs propres=[-2.1457  0.7457  1.    ]
                     etat fondamental=[0.5494 0.5494 0.6295] sans signe=True ecart spectral=2.8914 cousin - H = 0.0e+00
site 3 a +0.4 : a jauge pres=False 

### Lecture du résultat (4a) — ce que la définition garantit, et ce qu'elle ne garantit pas

**La conséquence est vérifiée.** Les trois matrices que la cellule déclare stoquastiques — la chaîne, le triangle, le cousin — ont un état fondamental à coefficients tous positifs ou nuls : $[0{,}5;\ 0{,}7071;\ 0{,}5]$, $[0{,}5774;\ 0{,}5774;\ 0{,}5774]$, $[0{,}5494;\ 0{,}5494;\ 0{,}6295]$. C'est la conséquence que l'article tire de la définition 2 : un état fondamental sans signe, donc interprétable comme une distribution de probabilité. La colonne « à jauge près » ne distingue rien ici, et c'est normal : ces trois matrices sont déjà stoquastiques telles quelles, la jauge identité suffit. La chaîne affiche un produit de boucle nul — il lui manque la liaison $0$–$2$, et un produit nul n'est pas un signe.

**La réciproque est fausse.** La boucle $(-,-,+)$ n'est stoquastique à aucune jauge (produit de boucle $+1$, les huit choix de signes échouent) ; son état fondamental a bien deux signes, $[-0{,}373;\ 0{,}373;\ 0{,}8496]$. Mais la **même** boucle avec le site 3 à $+0{,}4$ ou à $+1{,}0$ donne un état fondamental **sans signe** — $[0{,}7071;\ 0{,}7071;\ 0]$ — et reste, à toutes les jauges, hors de la classe. Un état fondamental sans signe n'est donc pas un certificat d'appartenance : la définition donne une condition **suffisante** pour cette conséquence, pas une caractérisation.

**Pourquoi le seuil tombe exactement à zéro.** Quel que soit $\varepsilon$, le vecteur $(1, 1, 0)/\sqrt2$ est vecteur propre exact d'énergie $-1$ : la première ligne de la matrice lui rend $-1$, la seconde aussi, et les deux termes opposés de la troisième s'annulent. Le reste du spectre vient du bloc antisymétrique couplé au site 3, et ce bloc passe sous $-1$ dès que $\varepsilon < 0$. Le fondamental bascule donc en $\varepsilon = 0$ indépendamment de l'amplitude de la frustration : la cellule le vérifie aux quatre valeurs $+0{,}4$, $+1{,}0$, $-0{,}4$, $-1{,}0$, avec à chaque fois $\varepsilon > 0 \Rightarrow$ sans signe et $\varepsilon < 0 \Rightarrow$ deux signes.

**Le prix de la contrainte, sur cette paire.** Le cousin de la boucle diffère d'elle de $2$ sur la seule liaison $(0,2)$, et cette différence suffit à faire passer l'énergie du fondamental de $-1{,}278$ à $-2{,}1457$, et l'écart spectral de $0{,}278$ à $2{,}8914$. La cellule 4a bis mesure cet effet sur deux mille matrices, et répond à la question que cette paire laisse ouverte : le cousin n'est-il pas, dans la classe, la matrice elle-même ?

In [20]:
# 4a bis. Deux mille matrices de quatre sites : la taille de la classe, et le prix des signes

paires = [(i, j) for i in range(4) for j in range(i + 1, 4)]
compte_signes = {"stoquastique": 0, "stoquastique a jauge pres": 0}
for motif in itertools.product((1.0, -1.0), repeat=6):       # enumeration exhaustive des 2^6 motifs de signes
    Hm = NP.zeros((4, 4))
    for (i, j), signe in zip(paires, motif):
        Hm[i, j] = Hm[j, i] = signe
    compte_signes["stoquastique"] += est_stoquastique(Hm)
    compte_signes["stoquastique a jauge pres"] += jauge_stoquastique(Hm) is not None
print("motifs de signes des 6 liaisons de K4, enumeration exhaustive :", compte_signes, "sur", 2 ** 6)

nb_echantillons = 2000
comptes = {"stoquastique": 0, "stoquastique a jauge pres": 0, "a jauge pres et sans signe": 0,
           "hors classe": 0, "hors classe et sans signe": 0}
ecarts_energie: dict[str, list[float]] = {"a jauge pres": [], "hors classe": []}
ecart_jauge_cousin = 0.0          # dans la classe : la matrice jaugee est-elle exactement son cousin ?
for _ in range(nb_echantillons):
    gaussienne = generateur.standard_normal((4, 4))
    Hm = (gaussienne + gaussienne.T) / 2            # symetrique reelle a coefficients continus
    jauge = jauge_stoquastique(Hm)
    fondamental, _ecart = etat_fondamental(Hm)
    cousin = cousin_stoquastique(Hm)
    cle = "a jauge pres" if jauge is not None else "hors classe"
    ecarts_energie[cle].append(float(NP.linalg.eigvalsh(cousin)[0] - NP.linalg.eigvalsh(Hm)[0]))
    if est_stoquastique(Hm):
        comptes["stoquastique"] += 1
    if jauge is not None:
        comptes["stoquastique a jauge pres"] += 1
        comptes["a jauge pres et sans signe"] += signe_libre(jauge * fondamental)
        ecart_jauge_cousin = max(ecart_jauge_cousin,
                                 float(NP.abs(jauge[:, None] * jauge[None, :] * Hm.real - cousin).max()))
    else:
        comptes["hors classe"] += 1
        comptes["hors classe et sans signe"] += signe_libre(fondamental)
print(f"sur {nb_echantillons} matrices symetriques reelles 4x4 a coefficients gaussiens :")
for cle, valeur in comptes.items():
    print(f"   {cle:32s} {valeur:5d}  ({100 * valeur / nb_echantillons:.1f} %)")
for cle, liste in ecarts_energie.items():
    print(f"   E0(cousin) - E0(H), {cle:14s} : min={min(liste):+.6f} moyenne={sum(liste) / len(liste):+.6f} "
          f"part strictement negative : {sum(1 for ecart in liste if ecart < -1e-9) / len(liste):.3f}")
print(f"   dans la classe, ecart max entre la matrice jaugee et son cousin : {ecart_jauge_cousin:.1e}")

motifs de signes des 6 liaisons de K4, enumeration exhaustive : {'stoquastique': 1, 'stoquastique a jauge pres': 8} sur 64


sur 2000 matrices symetriques reelles 4x4 a coefficients gaussiens :
   stoquastique                        33  (1.6 %)
   stoquastique a jauge pres          252  (12.6 %)
   a jauge pres et sans signe         252  (12.6 %)
   hors classe                       1748  (87.4 %)
   hors classe et sans signe          178  (8.9 %)
   E0(cousin) - E0(H), a jauge pres   : min=+0.000000 moyenne=+0.000000 part strictement negative : 0.000
   E0(cousin) - E0(H), hors classe    : min=-1.387775 moyenne=-0.234754 part strictement negative : 1.000
   dans la classe, ecart max entre la matrice jaugee et son cousin : 0.0e+00


### Lecture du résultat (4a bis) — une classe petite, et un cousin qui est la matrice elle-même

**Les deux comptages se répondent.** L'énumération des 64 motifs de signes de $K_4$ donne **un** motif directement stoquastique sur soixante-quatre, et **huit** à jauge près. Les 2000 matrices gaussiennes retrouvent les mêmes proportions : 33 directement (1,6 %), 252 à jauge près (12,6 %). Ce n'est pas une coïncidence : le motif de signes d'une matrice gaussienne symétrique est uniforme sur les 64, donc les proportions attendues sont exactement $1/64$ et $8/64$. La taille de la classe ne dépend pas du tirage, elle dépend du nombre de liaisons : chaque liaison coûte un facteur deux.

**Dans la classe, le cousin ne change rien — rien du tout.** Pour les 252 matrices à jauge près, l'écart entre la matrice jaugée et son cousin est de $0{,}0\times10^{0}$, et l'écart d'énergie $E_0(\text{cousin}) - E_0(H)$ vaut $+0{,}000000$ en minimum comme en moyenne, avec une part strictement négative de $0{,}000$. Autrement dit : hors du signe des vecteurs de base, le cousin **est** la matrice jaugée. Il n'y a pas deux objets, il y en a un.

**Hors de la classe, le cousin paie.** Pour les 1748 autres matrices, l'écart d'énergie est strictement négatif dans **tous** les cas : minimum $-1{,}387775$, moyenne $-0{,}234754$, part strictement négative $1{,}000$. Rendre tous les termes négatifs abaisse toujours l'énergie, et de beaucoup.

**Un état sans signe ne certifie rien.** 178 des 1748 matrices hors classe (10,2 %) ont elles aussi un état fondamental sans signe. La conséquence de la définition 2 n'est donc pas une caractérisation : pour savoir si une matrice est dans la classe, il faut regarder ses signes ou chercher une jauge, jamais son seul état fondamental.

**Ce que ce tirage ne dit pas.** Tout ceci est mesuré pour $n = 4$. La formule générale du nombre de motifs — $2^{n-1}$ sur $2^{n(n-1)/2}$ — est l'objet de l'exercice 3, qui la démontre puis l'énumère jusqu'à $n = 5$.

### Exercice 3 — Pourquoi huit motifs sur soixante-quatre

La cellule 4a bis a trouvé exactement un motif stoquastique sur les $64$ de $K_4$, et huit stoquastiques à jauge près. Le second nombre n'est pas une particularité de $K_4$ : il se généralise à tout graphe complet.

1. **Sur le papier.** Montrez qu'un motif de signes est stoquastique à jauge près si et seulement s'il existe des signes de sites $\sigma_1, \ldots, \sigma_n$ tels que $H_{ij} = -\sigma_i \sigma_j$ pour toute liaison $(i, j)$. Un sens est immédiat : prendre la jauge $s_i = \sigma_i$. Pour l'autre, posez $\sigma_1 = 1$ et propagez $\sigma_j = -H_{1j}$ : il faut que le résultat ne dépende pas du chemin suivi. Le produit des termes d'une boucle est invariant par jauge — les $s_i$ y apparaissent au carré — et c'est lui qui tranche.
2. **Le compte.** Il y a $2^n$ choix de $\sigma$, mais $\sigma$ et $-\sigma$ donnent le même motif. Comparez au total $2^{n(n-1)/2}$, puis à la part quand $n$ grandit.
3. **Par énumération.** Complétez `motifs_a_jauge` et vérifiez pour $n = 2, 3, 4, 5$.

In [21]:
# Exercice 3 : les motifs de signes stoquastiques a jauge pres sur le graphe complet K_n
def motifs_a_jauge(n: int) -> int:
    """Nombre de motifs de signes des n (n - 1) / 2 liaisons de K_n qui sont stoquastiques a jauge pres."""
    return None   # TODO etudiant
# Indice : une liaison est une paire (i, j) avec i < j ; un motif est un signe +1 ou -1 par liaison.
# Etape 1 : construire la matrice symetrique n x n d'un motif (voir la premiere boucle de la cellule 4a bis).
# Etape 2 : enumerer les 2**(n (n - 1) / 2) motifs et compter ceux pour lesquels jauge_stoquastique ne rend pas None.


for n in (2, 3, 4, 5):
    print(f"K_{n} : motifs stoquastiques a jauge pres = {motifs_a_jauge(n)}, "
          f"sur {2 ** (n * (n - 1) // 2)}")

K_2 : motifs stoquastiques a jauge pres = None, sur 2
K_3 : motifs stoquastiques a jauge pres = None, sur 8
K_4 : motifs stoquastiques a jauge pres = None, sur 64
K_5 : motifs stoquastiques a jauge pres = None, sur 1024


### Un vérificateur stoquastique face aux signes du prouveur

Le coût du second sens ne se mesure pas en énergie, mais en probabilité d'acceptation. Un vérificateur stoquastique ne fabrique aucun signe ; il en **reçoit** — ceux des amplitudes que le prouveur lui renvoie. La remarque 4.3 de Liu en donne l'exemple le plus court.

> **Remark 4.3 (Nonnegative joint states can be suboptimal)** — « Consider a verifier that prepares an EPR pair on (B, B″), sends B″ to the prover, and receives a qubit A. It swaps B with a fresh |0⟩ controlled on A = 1, then measures B in the Hadamard basis and accepts on +. For the joint state ρ on (B, A), let σb = (I ⊗ ⟨b|)ρ(I ⊗ |b⟩). The marginal condition gives σ0 + σ1 = I/2, so the acceptance probability is Tr(|+⟩⟨+|σ0) + ½ Tr(σ1) = ½ + ½ Tr(Xσ0) ≤ ¾. Here Tr(Xσ0) = ⟨+|σ0|+⟩ − ⟨−|σ0|−⟩ ≤ 1/2. Applying a Hadamard to B″ and returning it gives σ0 = |+⟩⟨+|/2 and attains ¾. If ρ is entrywise nonnegative in the computational basis, the off-diagonal entries of σ0 and σ1 are nonnegative and sum to zero. Then Tr(Xσ0) = 0, giving acceptance probability ½. »

Lire la remarque en trois temps.

- **La condition de marge.** $B$ est la moitié d'une paire préparée par le vérificateur. Quel que soit ce que le prouveur fait de l'autre moitié, et quel que soit le qubit $A$ qu'il renvoie, l'état de $B$ à lui seul reste $I/2$ : c'est $\sigma_0 + \sigma_1 = I/2$.
- **Le plafond.** Si $A = 1$, $B$ est remplacée par un $|0\rangle$ frais, qu'une lecture de Hadamard accepte une fois sur deux. Si $A = 0$, $B$ est lue telle quelle : l'acceptation dépend alors de ses termes hors diagonaux, et vaut au plus $1$.
- **Le coût.** Si l'état joint n'a que des amplitudes positives ou nulles, les deux termes hors diagonaux de $\sigma_0$ sont positifs et ceux de $\sigma_1$ aussi ; comme ils se compensent dans la somme $\sigma_0 + \sigma_1 = I/2$, ils sont nuls. La lecture de Hadamard ne voit plus rien : $1/2$.

Trois qubits suffisent pour rejouer ce vérificateur : $B$ (qubit 0), le qubit reçu $A$ (qubit 1), et l'auxiliaire frais (qubit 2). L'échange commandé par $A = 1$ se décompose en un CNOT, un Toffoli et un CNOT — trois portes de la définition 2.2.

La cellule 4b l'exécute sur trois prouveurs, puis sur $200$ prouveurs unitaires tirés au hasard, et contrôle chaque acceptation par deux formules indépendantes du circuit : celle de la remarque, $1/2 + \mathrm{Tr}(X \sigma_0)/2$, et une forme fermée qui ne dépend que de la première ligne de l'unitaire du prouveur. La cellule 4b bis isole ensuite les prouveurs **à états positifs**.

In [22]:
# 4b. Le deuxieme sens sur trois qubits : le jouet de la remarque 4.3

def acceptation_jouet(c: NP.ndarray) -> float:
    """Acceptation du verificateur de la remarque 4.3, sur l'etat pur d'amplitudes c[b, a] des qubits (B, A).

    B est la moitie de la paire gardee par le verificateur, A le qubit recu. Si A = 1, B est echange avec un
    auxiliaire frais dans l'etat |0> ; B est ensuite lue dans la base de Hadamard, et l'on accepte sur +.
    """
    etat = NP.zeros(8, dtype=complex)
    etat[:4] = c.T.ravel()                                  # indice = b + 2 a + 4 auxiliaire
    porte(etat, X, (2,), controles=(0,))                    # echange commande entre B et l'auxiliaire
    porte(etat, X, (0,), controles=(1, 2))
    porte(etat, X, (2,), controles=(0,))
    porte(etat, H, (0,))                                    # mesure de B dans la base de Hadamard
    return float((NP.abs(etat[0::2]) ** 2).sum())           # accepte si B donne 0 (issu de +)


def formule_4_3(c: NP.ndarray) -> float:
    """Le meme calcul par la formule de la remarque : 1/2 + 1/2 Tr(X sigma_0)."""
    sigma_0 = c[:, [0]] @ c[:, [0]].conj().T                # etat (non normalise) de B conditionne a A = 0
    return 0.5 + 0.5 * float(NP.real(NP.trace(X @ sigma_0)))


def forme_fermee_4_3(U: NP.ndarray) -> float:
    """Prouveur unitaire U sur la moitie envoyee : 1/2 + 1/2 Re(U[0, 1] conj(U[0, 0]))."""
    return 0.5 + 0.5 * float(NP.real(U[0, 1] * NP.conj(U[0, 0])))


etat_epr = NP.zeros(4, dtype=complex)
etat_epr[0] = etat_epr[3] = 1 / NP.sqrt(2)                  # registre (A, B) : A est le bit de poids fort
for nom, U in (("identite", NP.eye(2)), ("Hadamard", H), ("X", X)):
    c = (NP.kron(U, NP.eye(2)) @ etat_epr).reshape(2, 2).T   # c[b, a] : le prouveur agit sur la moitie recue
    print(f"prouveur {nom:9s} : c[b, a] = {NP.round(c.real, 4).tolist()} "
          f"acceptation du circuit = {acceptation_jouet(c):.12f} formule = {formule_4_3(c):.12f} "
          f"forme fermee = {forme_fermee_4_3(U):.12f}")
    print(f"{'':18s} B reste I/2 : {bool(NP.abs(c @ c.conj().T - 0.5 * NP.eye(2)).max() <= 1e-12)} ; "
          f"entrees reelles et positives : {bool((c.real >= -1e-12).all())}")
alea_haar = []
for _ in range(200):
    z = generateur.standard_normal((2, 2)) + 1j * generateur.standard_normal((2, 2))
    Qb, Rb = NP.linalg.qr(z)            # loi de Haar sur U(2) : factorisation QR d'une gaussienne complexe
    U = Qb @ NP.diag(NP.diag(Rb) / NP.abs(NP.diag(Rb)))
    c = (NP.kron(U, NP.eye(2)) @ etat_epr).reshape(2, 2).T
    alea_haar.append((acceptation_jouet(c), formule_4_3(c), forme_fermee_4_3(U)))
print(f"200 unitaires de Haar sur la moitie envoyee : acceptation max={max(a for a, _f, _g in alea_haar):.6f} "
      f"min={min(a for a, _f, _g in alea_haar):.6f} ; au-dessus de 3/4 : "
      f"{sum(1 for a, _f, _g in alea_haar if a > 0.75 + 1e-12)} ; "
      f"ecart max circuit/formule = {max(abs(a - f) for a, f, _g in alea_haar):.1e} ; "
      f"ecart max circuit/forme fermee = {max(abs(a - g) for a, _f, g in alea_haar):.1e}")

prouveur identite  : c[b, a] = [[0.7071, 0.0], [0.0, 0.7071]] acceptation du circuit = 0.500000000000 formule = 0.500000000000 forme fermee = 0.500000000000
                   B reste I/2 : True ; entrees reelles et positives : True
prouveur Hadamard  : c[b, a] = [[0.5, 0.5], [0.5, -0.5]] acceptation du circuit = 0.750000000000 formule = 0.750000000000 forme fermee = 0.750000000000
                   B reste I/2 : True ; entrees reelles et positives : False
prouveur X         : c[b, a] = [[0.0, 0.7071], [0.7071, 0.0]] acceptation du circuit = 0.500000000000 formule = 0.500000000000 forme fermee = 0.500000000000
                   B reste I/2 : True ; entrees reelles et positives : True
200 unitaires de Haar sur la moitie envoyee : acceptation max=0.749231 min=0.250717 ; au-dessus de 3/4 : 0 ; ecart max circuit/formule = 5.6e-16 ; ecart max circuit/forme fermee = 6.7e-16


### Lecture du résultat (4b) — d'où vient le plafond de $3/4$

**Trois prouveurs, trois nombres, trois voies d'accord.** Identité et $X$ donnent $1/2$, Hadamard donne $3/4$. Le circuit, la formule de la remarque et la forme fermée coïncident sur les trois, et la marge $c\,c^\dagger = I/2$ tient partout : c'est bien le même nombre, calculé trois fois.

**Ce qui distingue Hadamard n'est pas « avoir des signes », c'est le module.** Son état a une entrée négative ($-0{,}5$) et atteint $3/4$ ; identité et $X$ ont des états à entrées positives ou nulles et plafonnent à $1/2$. La forme fermée dit exactement pourquoi : $p = 1/2 + \frac12\operatorname{Re}(U_{01}\overline{U_{00}})$, majoré par $1/2 + \frac12\lvert U_{00}\rvert\lvert U_{01}\rvert \le 3/4$. L'égalité demande **deux** choses à la fois — les deux entrées de la première ligne de l'unitaire du prouveur de module $1/\sqrt2$, et de même phase. L'unité de Hadamard est le cas le plus simple de ce couple ; une matrice de permutation a $U_{01} = 0$ et reste à $1/2$.

**Le plafond tient sur 200 prouveurs quelconques** : maximum $0{,}749231$, zéro au-dessus de $3/4$. Le minimum, lui, descend à $0{,}250717$ — la borne de la remarque est unilatérale, un prouveur maladroit fait moins bien que $1/2$. Ce que la cellule vérifie est le plafond sur un échantillon de 200 ; le théorème, lui, quantifie sur tous les prouveurs, et cette cellule ne le démontre pas.

**L'écart entre les voies** est de $5{,}6\times10^{-16}$ (circuit contre formule) et de $6{,}7\times10^{-16}$ (circuit contre forme fermée) : trois écritures indépendantes du même nombre, à l'arithmétique flottante près.

In [23]:
# 4b bis. Le prix de la contrainte sur ce jouet : prouveurs a etats positifs, avec et sans registre prive

PERMUTATIONS = ((NP.eye(2, dtype=complex), "identite"), (X, "echange"))
monomiales = [P @ NP.diag(signes) for P, _nom in PERMUTATIONS
              for signes in itertools.product((1, -1), repeat=2)]
print(f"matrices 2x2 reelles monomiales : {len(monomiales)} ; unitaires : "
      f"{sum(bool(NP.abs(M @ M.T - NP.eye(2)).max() <= 1e-12) for M in monomiales)} ; a entrees positives : "
      f"{sum(bool(((M.real >= 0) & (NP.abs(M.imag) <= 1e-12)).all()) for M in monomiales)}")
for P, nom in PERMUTATIONS:
    c = P.T / NP.sqrt(2)        # etat joint pur, atteignable et positif : c[b, a] = P[a, b] / racine de 2
    print(f"prouveur permutation {nom:8s} : c = {NP.round(c.real, 4).tolist()} ; "
          f"c c^dagger - I/2 : ecart max = {float(NP.abs(c @ c.conj().T - 0.5 * NP.eye(2)).max()):.1e} ; "
          f"acceptation du circuit = {acceptation_jouet(c):.12f}")
alea_positifs = []
for _ in range(200):
    poids = generateur.dirichlet(NP.ones(3))         # melange avec registre prive : rho = somme_q |c_q><c_q|
    c_q = [NP.sqrt(l) * PERMUTATIONS[generateur.integers(2)][0] / NP.sqrt(2) for l in poids]
    alea_positifs.append(sum(acceptation_jouet(c) for c in c_q))
print(f"200 melanges avec registre prive de prouveurs a etats positifs : acceptation "
      f"min={min(alea_positifs):.12f} max={max(alea_positifs):.12f} ; prix de la contrainte sur ce jouet, "
      f"3/4 - moyenne = {0.75 - sum(alea_positifs) / len(alea_positifs):.12f}")
alea_sans_marge = []
for _ in range(200):
    theta, phi = generateur.uniform(0, NP.pi / 2, size=2)
    c = NP.array([[NP.cos(theta), NP.sin(theta)], [NP.sin(phi), NP.cos(phi)]]) / NP.sqrt(2)
    alea_sans_marge.append((acceptation_jouet(c), float(NP.abs(c @ c.conj().T - 0.5 * NP.eye(2)).max())))
print(f"200 etats joints a entrees positives, SANS la contrainte sur la marginale de B : acceptation "
      f"min={min(a for a, _m in alea_sans_marge):.12f} max={max(a for a, _m in alea_sans_marge):.12f} ; "
      f"ecart max de c c^dagger a I/2 = {max(m for _a, m in alea_sans_marge):.6f}")

matrices 2x2 reelles monomiales : 8 ; unitaires : 8 ; a entrees positives : 2
prouveur permutation identite : c = [[0.7071, 0.0], [0.0, 0.7071]] ; c c^dagger - I/2 : ecart max = 1.1e-16 ; acceptation du circuit = 0.500000000000
prouveur permutation echange  : c = [[0.0, 0.7071], [0.7071, 0.0]] ; c c^dagger - I/2 : ecart max = 1.1e-16 ; acceptation du circuit = 0.500000000000
200 melanges avec registre prive de prouveurs a etats positifs : acceptation min=0.500000000000 max=0.500000000000 ; prix de la contrainte sur ce jouet, 3/4 - moyenne = 0.250000000000
200 etats joints a entrees positives, SANS la contrainte sur la marginale de B : acceptation min=0.500125926336 max=0.998563896780 ; ecart max de c c^dagger a I/2 = 0.500000


### Lecture du résultat (4b bis) — le prix, chiffré : un quart

**Deux états purs positifs, pas un de plus.** Sur les huit matrices monomiales réelles $2\times2$, deux ont toutes leurs entrées positives ou nulles : l'identité et $X$. Ce n'est pas une particularité de l'échantillon, et la raison est courte. La marge impose $c\,c^\dagger = I/2$, donc l'orthogonalité des deux colonnes de $c$ ; si les entrées sont positives ou nulles, deux entrées d'une même ligne ne peuvent pas être toutes deux non nulles, et le support se réduit à une diagonale ou à une anti-diagonale. Il ne reste que $I/\sqrt2$ et $X/\sqrt2$ — deux états purs, tous deux à $1/2$, avec un écart de marge de $1{,}1\times10^{-16}$.

**Le prix est exactement un quart, et il ne bouge pas.** Les 200 mélanges de ces deux états purs — c'est-à-dire les prouveurs à état joint positif qui respectent la marge, le registre privé ne changeant que les poids — donnent $1/2$ en minimum comme en maximum, et le prix mesuré $3/4 - 1/2$ vaut $0{,}250000000000$. Sur ce jouet, la contrainte ne coûte pas « un peu » : elle coûte un quart, pour tous les prouveurs essayés.

**Ce qui plafonne est le couple (positivité, marge), pas la positivité seule.** Si l'on retire la contrainte sur la marginale de $B$, les mêmes états à entrées positives montent jusqu'à $0{,}998564$, avec un écart de $c\,c^\dagger$ à $I/2$ de $0{,}5$. C'est exactement le raisonnement de la remarque : la marge $\sigma_0 + \sigma_1 = I/2$ et des termes croisés positifs qui doivent se compenser un à un, donc s'annuler.

**À ne pas confondre avec le témoin de 2006.** La remarque de Bravyi, Bessen et Terhal sur le témoin non négatif porte sur un vérificateur **non interactif**, où choisir un état positif ne coûte rien. Ici l'état qui compte est **joint** entre le vérificateur et le prouveur : sa positivité est une contrainte sur un objet corrélé, et elle est sous-optimale. Les deux énoncés ne se contredisent pas ; ils ne parlent pas du même objet.

In [24]:
# 4c. Les signes que porte reellement le circuit a deux messages, sur l'instance vraie de la section 3
for nom in ("honnete (p du Lemme 3.7)", "aleatoire 1", "aleatoire 2"):
    avant_test, etat = verif.deroule(strategies[nom])
    print(f"{nom:28s} partie imaginaire max : avant le test = {float(NP.abs(avant_test.imag).max()):.1e}, "
          f"apres = {float(NP.abs(etat.imag).max()):.1e} ; amplitude reelle la plus petite apres le test = "
          f"{float(etat.real.min()):+.6f} ; norme = {float(NP.linalg.norm(etat)):.12f}")
print(f"registre du circuit : {int(round(NP.log2(etat.size)))} qubits, simules classiquement par leur "
      f"vecteur d'etat de taille {etat.size}")
print(f"duree totale du carnet : {time.perf_counter() - CHRONO:.1f} s")

honnete (p du Lemme 3.7)     partie imaginaire max : avant le test = 0.0e+00, apres = 0.0e+00 ; amplitude reelle la plus petite apres le test = +0.000000 ; norme = 1.000000000000


aleatoire 1                  partie imaginaire max : avant le test = 0.0e+00, apres = 0.0e+00 ; amplitude reelle la plus petite apres le test = -0.039804 ; norme = 1.000000000000


aleatoire 2                  partie imaginaire max : avant le test = 0.0e+00, apres = 0.0e+00 ; amplitude reelle la plus petite apres le test = -0.040576 ; norme = 1.000000000000
registre du circuit : 19 qubits, simules classiquement par leur vecteur d'etat de taille 524288
duree totale du carnet : 131.5 s


### Lecture du résultat (4c) — les signes que le circuit porte vraiment

**Aucune partie imaginaire, nulle part.** Sur les 19 qubits — 524 288 amplitudes — la partie imaginaire maximale est nulle avant comme après le test, pour les trois prouveurs. Les portes du vérificateur ($H$, CNOT, Toffoli) sont réelles et les isométries construites par la preuve le sont aussi : le circuit ne fabrique aucun signe complexe, et le jouet ne doit rien à une convention de phase.

**Le prouveur honnête est positif, les autres non.** L'amplitude réelle la plus petite s'arrête à $+0{,}000000$ pour le prouveur honnête : toutes ses amplitudes sont positives ou nulles. Les deux prouveurs aléatoires descendent à $-0{,}039804$ et $-0{,}040576$ : le circuit transporte leurs signes jusqu'au test de SWAP sans les corriger.

**Ce que la cellule ne dit pas.** Que le prouveur honnête soit positif sur cette instance ne signifie pas que le prouveur optimal le soit en général — c'est précisément ce que la remarque 4.3 interdit de supposer, et la cellule 4b l'a mesuré sur un jouet à trois qubits. La cellule ne dit pas non plus que trois prouveurs épuisent la classe : elle décrit un état à un instant donné du protocole, pour trois isométries choisies, sur 19 qubits simulés classiquement.

## 5. Mesuré, cité, ouvert

Trois colonnes, et rien ne passe de l'une à l'autre sans être dit. Ce qui est mesuré se rejoue en exécutant ce carnet ; ce qui est cité ne s'y rejoue pas ; ce qui est ouvert s'écrit comme ouvert.

### Mesuré

- **Le sum-check, tour par tour** : quatre formules jouets sur $\mathbb{F}_5$, les degrés individuels (maximum $2$, jamais $3$), neuf messages pour trois tours, et le prouveur honnête accepté sur **toutes** les suites de défis des formules vraies ($2$, $3$, $5$, $7$ à une variable ; $8$, $27$, $125$, $343$ à deux), sur aucune de celles des formules fausses. Le meilleur prouveur trompeur est calculé par programmation dynamique **exacte** — pas par tirage : $0{,}400000$ à $q = 5$ pour $T = 1$, et les valeurs pour $T = 3$.
- **L'échantillonneur et l'équation (3.5)** : sommes des colonnes à $1$ avec un écart d'au plus $1{,}0\times10^{-16}$ sur douze combinaisons d'instances et de corps, la distance en variation totale de la loi honnête égale à sa valeur attendue, et le minimum sur **toutes** les lois d'entrée calculé par programme linéaire ($0{,}047619$ à $q = 3$, $0{,}047000$ à $q = 5$, $0{,}043064$ à $q = 7$ sur l'instance fausse à une variable).
- **Les deux bornes de l'article, confrontées au calcul** : la borne $2T/q$ du sum-check, atteinte exactement par le meilleur trompeur à $T = 1$ pour $q \ge 3$ — et non à $q = 2$, où elle vaut $1$ et ne contraint rien —, jamais à $T = 3$, où il reste strictement en dessous ; la borne $1/(4q(T+1))$ du Lemme 3.8, tenue sur les quatre lignes du régime $q > 3(T+1)$, tenue encore hors régime sur deux lignes ($(m,q) = (1,3)$ et $(1,5)$) et **mise en défaut** sur quatre des douze lignes hors régime — $(m,q) = (1,2)$, $(2,2)$, $(2,3)$, $(2,5)$, toutes sur l'instance fausse. La preuve ne s'y applique pas, et la borne n'y tient pas : vérifier une borne, c'est aussi mesurer où elle tombe.
- **Le protocole à deux messages, simulé qubit par qubit** : $19$ qubits, l'acceptation littérale, celle de l'équation (4.1) et celle de la formule réduite d'accord à $8{,}9\times10^{-16}$ sur cinq prouveurs ; l'effet d'une copie gardée par le prouveur, qui fait tomber l'honnête de $1{,}000000$ à $0{,}718750$ ; l'instance fausse à $q = 2$, hors promesse, où le prouveur honnête n'obtient que $0{,}864277$ ; le balayage de $q$ avec le compte de copies du lemme 4.2 ($78$, $76$, $75$, puis « infini »).
- **Les deux sens de « stoquastique »** : la conséquence de la définition 2 sur les matrices de trois sites, et l'absence de réciproque — une boucle frustrée hors classe à état fondamental signé, deux matrices hors classe à état fondamental sans signe ; le cousin égal à la matrice jaugée dans la classe ($0{,}0\times10^{0}$) et strictement plus bas hors de la classe ($100\,\%$ des $1748$ cas) ; la taille de la classe ($1$ motif sur $64$, $8$ à jauge près, $1{,}6\,\%$ et $12{,}6\,\%$ sur $2000$ matrices).
- **Le coût du second sens** : $1/2$, $3/4$, $1/2$ pour identité, Hadamard et $X$ ; $200$ prouveurs de Haar sous le plafond (maximum $0{,}749231$) ; le prix de la positivité mesuré à $0{,}250000000000$ sur $200$ prouveurs à marge correcte, et jusqu'à $0{,}998564$ quand on retire la marge de $B$.
- **Les signes que le circuit porte** : partie imaginaire nulle sur les $524\,288$ amplitudes, prouveur honnête d'amplitude minimale $+0{,}000000$, prouveurs aléatoires à $-0{,}039804$ et $-0{,}040576$.

### Cité

- $QIP(3) = QIP = PSPACE$ — Jain, Ji, Upadhyay & Watrous, *Journal of the ACM* 58(6), 2011. Établi depuis 2011 ; cité ici, **jamais rejoué**.
- $qq\text{-}\mathrm{StoqAM} = qq\text{-}\mathrm{QAM} = QIP(2) = QIP = PSPACE$, la compression du protocole à trois messages en deux, le théorème 4.4 et la difficulté PSPACE de `CIUniform` — Yupan Liu, arXiv 2611.31236**v1**, **preprint du 23 novembre 2026**, auteur unique, sans relecture par les pairs, sans version 2 à la date d'écriture de ce carnet. L'article cite une déclaration d'usage d'outils d'intelligence artificielle.
- Les définitions 2 et 4, la classe `StoqMA`, l'inclusion $MA \subseteq \mathrm{StoqMA} \subseteq SBP \cap QMA$ et l'impossibilité d'amplifier par vote majoritaire — Bravyi, Bessen & Terhal, arXiv quant-ph/0611021v2.
- $IP = PSPACE$ (Lund–Fortnow–Karloff–Nisan 1992 ; Shamir 1992), le sum-check de Shamir et de Shen (1992) et la parallélisation à trois messages $QIP(3) = QIP$ (Kitaev–Watrous, STOC 2000), cités au §0 et lus dans le texte de Liu, pas dans les articles originaux.

### Ouvert, et ce que ce carnet ne fait pas

- **Aucune relecture par les pairs** : à aucun moment ce carnet ne présente $QIP(2) = PSPACE$ comme démontré indépendamment. Il rejoue le protocole et mesure ; la preuve est celle d'un preprint daté, d'un seul auteur.
- $U_S^\dagger$ est appliquée comme une permutation de registres, pas décomposée en portes.
- Le circuit qubit par qubit n'est simulé qu'à $q = 2$ ; partout ailleurs c'est l'arithmétique sur $\mathbb{F}_q$ qui est évaluée, alors que l'article choisit $q = 8(T+1)$.
- Les lignes vides des isométries sont complétées par un choix d'implémentation, explicité au §3e — et l'exercice 2 mesure ce que ce choix change.
- La solidité est **vérifiée sur les prouveurs testés**, pas démontrée : le théorème quantifie sur tous les prouveurs.
- Le prouveur du protocole est illimité ; rien ici ne borne son temps de calcul.
- Les comptages de rareté valent pour $n = 4$ ; la formule $2^{n-1}$ est démontrée à l'exercice 3, pas mesurée pour tout $n$.
- $19$ qubits sont simulés classiquement : c'est un jouet, pas une instance.

## 6. Place dans la série

Ce carnet prolonge une lignée qui va du temps au quantique, en passant par les oracles et la simulation.

- [03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-Python.ipynb) a posé la hiérarchie en temps : ce qu'une machine gagne quand on lui donne davantage de temps.
- [03c — Naviguer le Zoo exécutable](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) a lu la carte des classes et montré, oracles à l'appui, que deux mondes peuvent différer sans qu'aucune preuve ne tranche ; c'est là qu'a été posée la distinction entre ce qui est **exécutable**, ce qui est **cité** et ce qui reste **ouvert**.
- [06 — Simuler un circuit quantique classiquement](Complexity-06-Simuler-Circuit-Quantique-Classiquement-Python.ipynb) a simulé un circuit porte par porte et situé la barrière : la simulation naïve est exponentielle, le formalisme de Clifford reste polynomial.
- [06b — Aaronson & Gottesman, déquantification](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb) a exécuté le simulateur CHP et montré ce qu'une famille de circuits garde de classique.
- **03j** apporte la brique qui manquait : la **preuve interactive**. Le vérificateur n'y calcule plus seul, il interroge un prouveur ; le nombre de messages devient une ressource, et c'est cette ressource que 2011 puis l'annonce de 2026 rognent.

Deux fils relient ce carnet à ses aînés. Le **sum-check** du §1 est un cousin du vérificateur de 03 : une machine qui refuse de tout recalculer et délègue la vérification à des identités polynomiales. Et le **circuit** du §3 est un cas particulier de ce que 06 simule — $19$ qubits, des portes réelles, un vecteur d'état de $524\,288$ amplitudes —, avec une différence : la lecture se fait dans la base de Hadamard, et c'est là que se joue la définition de « stoquastique ». Enfin, la distinction mesuré / cité / ouvert héritée de 03c trouve ici son cas le plus serré : un résultat annoncé trois jours avant l'écriture de ce carnet.

## 7. Conclusion

- **Trois messages suffisent.** $QIP(3) = QIP = PSPACE$, Jain, Ji, Upadhyay & Watrous, 2011 : établi, et cité ici sans être rejoué.
- **Deux messages suffiraient.** Annoncé le 23 novembre 2026 par un preprint d'un seul auteur, non évalué par les pairs. Ce carnet n'en reproduit pas la preuve ; il exécute le protocole que cette preuve construit, sur des jouets, et trouve ce que le protocole promet : complétude parfaite sur l'instance vraie ($1{,}000000000000$ par trois voies indépendantes), une solidité que les prouveurs testés ne franchissent pas, et une marge d'un pour cent environ, que la répétition parallèle doit creuser.
- **La compression change la nature du vérificateur.** Passer de trois messages à deux n'enlève pas un tour : le vérificateur n'envoie plus que des moitiés de paires EPR, et ne fait plus qu'une vérification stoquastique. La différence structurelle se voit à l'exécution — un test de SWAP, $19$ qubits, une acceptation qui ne dépend que de la loi du prouveur et de son image — et impose une contrainte au prouveur : rendre la référence quantique sans en garder de copie. Le prouveur honnête lui-même, s'il en garde une, tombe de $1{,}000000$ à $0{,}718750$.
- **« Stoquastique » coûte, et le carnet le chiffre.** Une matrice stoquastique a un état fondamental sans signe, la réciproque est fausse, et la classe est petite : huit motifs sur soixante-quatre à jauge près. Côté vérificateur, un prouveur à état joint positif respectant la marge ne dépasse pas $1/2$ là qu'un prouveur quelconque atteint $3/4$ : un quart perdu, sur ce jouet.

Ce carnet ne dit pas que $QIP(2) = PSPACE$ est vrai. Il dit ce que le protocole fait quand on l'exécute : où il accepte à coup sûr, où il tombe à $1/2$, ce qu'un prouveur perd à garder une copie, et pourquoi un vérificateur qui ne peut pas produire de signe subit ceux du prouveur. C'est ce qu'une simulation peut dire d'une annonce : la part qui en est exécutable, mesurée — et datée.

## Références

Deux statuts de lecture, qui ne se confondent pas. Les quatre premières publications ont une **copie archivée** au gisement `Bibliographie IA/Automata/Complexity Theory and Hardness`, et chaque attribution de ce carnet se lit contre cette copie. Les quatre suivantes n'en ont pas à la date d'écriture : elles sont citées d'après la bibliographie de Liu, concordante avec l'enregistrement Crossref de leur DOI, et **non lues en copie** — rien dans ce carnet ne dépend de leur contenu au-delà de ce que Liu en dit. Aucune publication n'est commitée dans le dépôt.

- **Yupan Liu**, *Two messages suffice for quantum interactive proof systems*, arXiv [2611.31236](https://arxiv.org/abs/2611.31236) v1, 23 novembre 2026 — **preprint, auteur unique, non évalué par les pairs** ; l'article porte une déclaration d'usage d'outils d'intelligence artificielle. Copie archivée : `2026 - Liu - Two messages suffice for quantum interactive proof systems (arXiv 2611.31236v1, preprint du 23 novembre 2026, auteur unique).pdf`. Cités : les définitions 2.2, 3.1 et 4.1, le lemme 2.3, les lemmes 3.5 à 3.8, le lemme 4.2, la remarque 4.3, le théorème 4.4, le protocole 2 et les équations (3.4), (3.5), (4.1).
- **Sergey Bravyi, Arvid J. Bessen, Barbara M. Terhal**, *Merlin-Arthur Games and Stoquastic Complexity*, arXiv [quant-ph/0611021](https://arxiv.org/abs/quant-ph/0611021) v2, 19 décembre 2006 (preprint, telle que Liu le cite). Copie archivée : `2006 - Bravyi Bessen Terhal - Merlin-Arthur Games and Stoquastic Complexity (arXiv quant-ph 0611021v2, preprint du 19 decembre 2006).pdf`. Cités : les définitions 2 et 4, l'inclusion $MA \subseteq \mathrm{StoqMA} \subseteq SBP \cap QMA$ (résumé), le théorème 3, la remarque sur le témoin qui maximise l'acceptation et l'impossibilité d'amplifier par vote majoritaire.
- **Scott Aaronson**, billet du 25 novembre 2026 sur son blog (`scottaaronson.blog`, `p=10379`), qui signale le preprint de Liu. Copie archivée : `2026 - Aaronson - HOLY SHIT HOLY SHIT HOLY SHIT HOLY SHIT (scottaaronson.blog p=10379, billet du 25 novembre 2026, cite Liu arXiv 2611.31236).html`. Ce carnet ne reprend de ce billet aucune affirmation technique : ses nombres viennent du preprint et de l'exécution.
- **Rahul Jain, Zhengfeng Ji, Sarvagya Upadhyay, John Watrous**, *QIP = PSPACE*, *Journal of the ACM* 58(6):30:1–30:27, 2011, doi:10.1145/2049697.2049704 ; version préliminaire STOC 2010 ; arXiv:0907.4737. Copie archivée (le preprint) : `2009 - Jain Ji Upadhyay Watrous - QIP = PSPACE (arXiv 0907.4737v2, preprint du 3 aout 2009).pdf`. Établi ; cité, **non rejoué** : sa preuve est la méthode des poids multiplicatifs sur des programmes semidéfinis, qu'aucune cellule de ce carnet n'exécute.
- **Alexei Kitaev, John Watrous**, *Parallelization, amplification, and exponential time simulation of quantum interactive proof systems*, STOC 2000, pages 608–617, doi:10.1145/335305.335387 — la parallélisation à trois messages.
- **Carsten Lund, Lance Fortnow, Howard Karloff, Noam Nisan**, *Algebraic methods for interactive proof systems*, *Journal of the ACM* 39(4):859–868, 1992, doi:10.1145/146585.146605 ; **Adi Shamir**, *IP = PSPACE*, *Journal of the ACM* 39(4):869–877, 1992, doi:10.1145/146585.146609 ; **Alexander Shen**, *IP = PSPACE: Simplified proof*, *Journal of the ACM* 39(4):878–880, 1992, doi:10.1145/146585.146613 (titre et auteur tels que Liu les donne ; l'enregistrement Crossref de ce dernier DOI porte « IP = SPACE », A. Shen). C'est au protocole de Shamir et de Shen que Liu attribue le sum-check, rejoué au §1 sous sa forme à tours.

## À retenir

1. **Deux statuts, jamais confondus.** $QIP(3) = QIP = PSPACE$ est établi (Jain, Ji, Upadhyay & Watrous, 2011) ; $QIP(2) = PSPACE$ est annoncé (Yupan Liu, arXiv `2611.31236v1`, 23 novembre 2026) : un preprint d'un seul auteur, sans relecture par les pairs à la date d'écriture. Ce carnet rejoue le protocole du second et cite le premier ; il ne démontre ni l'un ni l'autre.
2. **Le sum-check se mesure avant de se comprimer.** Sur $\mathbb{F}_5$, trois tours, neuf messages : le prouveur honnête passe toutes les suites de défis d'une formule vraie, et le meilleur trompeur plafonne à $2T/q$ — borne atteinte à $T = 1$ pour $q \ge 3$ ($0{,}400000$ à $q = 5$), jamais à $T = 3$.
3. **Deux messages, ce n'est pas trois messages moins un.** Le vérificateur envoie des moitiés de paires EPR et lit le qubit de sortie d'un test de SWAP dans la base de Hadamard. Le prix se paie en cohérence : un prouveur qui garde une copie de la valeur reçue sépare les secteurs qui devaient interférer, et l'honnête lui-même tombe de $1{,}000000$ à $0{,}718750$.
4. **« Stoquastique » a deux sens.** Pour une matrice : des termes réels négatifs ou nuls hors diagonale — son état fondamental est alors sans signe, sans réciproque. Pour un vérificateur : des portes X, CNOT et Toffoli, une mesure de $|+\rangle$ — et l'état joint non négatif n'y est pas toujours le meilleur (remarque 4.3).
5. **La contrainte se chiffre.** Sur le jouet de la remarque 4.3, un prouveur à état joint positif qui respecte la marge de $B$ plafonne à $1/2$, là où un prouveur quelconque atteint $3/4$ : un quart perdu, $0{,}250000000000$ mesuré. Et la solidité du protocole complet n'est qu'une marge d'un pour cent : il faut $78$, $76$ ou $75$ copies, selon le corps, pour passer sous $3/4$.

## Vérifiez votre compréhension

1. Un protocole à trois tours échange combien de messages ? À $T = 1$, le meilleur prouveur trompeur atteint-il la borne $2T/q$ sur une instance fausse ? Et à $T = 3$ ?
2. Que dit le Lemme 3.7 de la loi honnête, et à quelle condition le Lemme 3.8 borne-t-il la distance à l'uniforme de *toute* loi d'entrée ? Où cette seconde borne tombe-t-elle sur les jouets ?
3. Un prouveur du protocole à deux messages garde une copie de la valeur reçue. Quelle acceptation obtient-il avec l'isométrie de la loi honnête, et pourquoi ?
4. Une matrice réelle symétrique a un état fondamental à coefficients tous positifs ou nuls. Est-elle stoquastique, ne serait-ce qu'à jauge de signes près ? Quel est le critère à vérifier ?
5. Dans le jouet de la remarque 4.3, que perd un prouveur à état joint positif qui respecte la marge de $B$ ? Quelles sont les deux contraintes dont la conjonction produit cette perte ?

<details><summary>Réponses</summary>

1. Neuf messages, trois par tour. À $T = 1$, le meilleur trompeur atteint $2T/q$ exactement pour $q \ge 3$ : $0{,}400000$ dans $\mathbb{F}_5$. À $q = 2$, la borne vaut $1$ et ne contraint rien ; il obtient $0{,}500000$. À $T = 3$, il reste strictement en dessous : $0{,}784000$ pour une borne de $1{,}200000$ dans $\mathbb{F}_5$.
2. Lemme 3.7 : sur une formule vraie, l'image de la loi honnête est uniforme ($TV = 0$) ; sur une formule fausse, elle en est à distance $1/(q(T+1))$. Lemme 3.8 : sous la condition $q > 3(T+1)$, toute loi d'entrée est à distance au moins $1/(4q(T+1))$ de l'uniforme. Sur les jouets, la borne tient sur les quatre lignes du régime et tombe sur quatre des douze lignes hors régime — $(m,q) = (1,2)$, $(2,2)$, $(2,3)$, $(2,5)$ — chaque fois sur l'instance fausse.
3. $0{,}718750$, au lieu de $1$. La copie range la valeur reçue dans un registre privé : les huit valeurs reçues coexistent dans des secteurs orthogonaux, la somme cohérente qui restituait $\lvert+\rangle_R$ devient une somme de probabilités, et le recouvrement avec la référence tombe de $1$ à $0{,}437500$. L'acceptation vaut $(1 + 0{,}4375)/2$.
4. Pas nécessairement : la définition donne une condition suffisante, pas une caractérisation. La boucle $(-,-,+)$ avec le site 3 à $+0{,}4$ a un état fondamental sans signe et n'est stoquastique à aucune jauge. Le critère : il existe des signes $s_i = \pm 1$ tels que $s_i s_j H_{ij} \le 0$ pour tous $i \ne j$. Le produit des termes d'une boucle y est invariant (les signes $s_i$ y entrent au carré) : sur un triangle, un produit positif exclut la classe.
5. Un quart : il plafonne à $1/2$, là où un prouveur quelconque atteint $3/4$. Les deux contraintes sont la positivité des amplitudes de l'état joint et la marge de $B$, $\sigma_0 + \sigma_1 = I/2$ : des termes hors diagonaux positifs qui doivent se compenser un à un sont nuls, et la lecture de Hadamard ne voit plus rien. Sans la marge, un état joint positif monte jusqu'à $0{,}998564$ : c'est le couple qui coûte, pas la positivité seule.

</details>

## Pour aller plus loin

- **Relire la base établie à la source** : Rahul Jain, Zhengfeng Ji, Sarvagya Upadhyay & John Watrous, *QIP = PSPACE*, arXiv [0907.4737](https://arxiv.org/abs/0907.4737) (2009 ; *Journal of the ACM* 58(6), 2011) — copie archivée au gisement `Bibliographie IA/Automata/Complexity Theory and Hardness`. Ce carnet n'en rejoue rien : sa preuve est la méthode des poids multiplicatifs sur des programmes semidéfinis.
- **Relire l'annonce en première version** : Yupan Liu, *Two messages suffice for quantum interactive proof systems*, arXiv [2611.31236](https://arxiv.org/abs/2611.31236) v1 (23 novembre 2026) — copie archivée au même gisement. Toute attribution du carnet se lit contre cette copie ; une version 2, une relecture par les pairs ou une formalisation feraient de la section 5 un état daté.
- **Ce que le carnet laisse à écrire** : la décomposition de $U_S^\dagger$ en portes, la répétition en série du protocole à deux messages, et la comparaison entre la répétition parallèle du lemme 4.2 et le vote majoritaire, dont Bravyi, Bessen et Terhal établissent l'impossibilité pour un vérificateur stoquastique.
- **Versions** : ce carnet n'a, à la date d'écriture, ni jumeau C#, ni jumeau Rust, ni version `_en`.

**Navigation** : [Index de la série](README.md) | [03h — L'exposant du produit matriciel est 9/4](Complexity-03h-ExposantProduitMatriciel-NineQuarts-Python.ipynb) | [03k — L'arbre couvrant de poids minimal en temps linéaire déterministe](Complexity-03k-KhannaPuttermanSong-ArbreCouvrantLineaireDeterministe-Python.ipynb) | [03 — Plus de temps, plus de problèmes](Complexity-03-HierarchieTemporelle-Python.ipynb) | [03c — Zoo et oracles](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) | [06 — Simuler un circuit quantique classiquement](Complexity-06-Simuler-Circuit-Quantique-Classiquement-Python.ipynb) | [06b — Aaronson & Gottesman, déquantification](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb)